# GSD — train trên B-Free `SD2.1_inpainted_samecat_origBG` (kế hoạch P2)

**Input duy nhất:** [baopv051/b-free-training-data](https://www.kaggle.com/datasets/baopv051/b-free-training-data).
**Bật:** GPU (T4 hoặc P100, chỉ dùng 1 GPU) và **Internet** (tải CLIP ViT-L/14 pretrained).
Rồi bấm **Save Version → Save & Run All**. Notebook tự chứa toàn bộ code.

## Notebook làm gì
1. **Audit** (~10–15 phút, CPU): đọc `train_list.csv`/`valid_list.csv` (kiểm MD5 chính thức), lấy
   `train_list − valid_list` = 41.213 COCO ID. Giải mã từng cặp `COCO_real_512` / `samecat_origBG`,
   tính mask vùng inpaint = `max kênh |fake − real| > 8`. Loại cặp lỗi/thiếu, mask < 0,1% hoặc > 95%.
   Lưu `audit_train.csv`, `audit_summary.json`; mask của 10.304 ID valid lưu riêng để phân tích sau.
2. **Chia:** 2.000 ID val (phân tầng theo 3 nhóm diện tích mask), phần còn lại xáo 1 lần cố định.
   **Virtual epoch (VE) = 12.600 ID = 25.200 ảnh** (đúng cỡ 1 epoch GSD-aug-Tiny), tối đa 3 VE, không lặp ID.
   **Không** dùng `valid_list` và **không** dùng 5 nhánh B-Free còn lại để chọn checkpoint.
3. **Train:** batch 8 = 4 COCO ID × (real, fake) → basis GSD tính trên cặp cùng nội dung.
   Blur (p0.5, σ∈[0,2]) → JPEG (p0.5, Q75–100, 4:4:4) **dùng chung tham số cho cả cặp**, đổi mỗi VE.
   Không flip/crop/resize. Model, GSD (4 block cuối, k=64), AdamW lr 1e-6, fp16, crop560/pad14:
   **giống hệt GSD-aug-Tiny**, khởi tạo từ CLIP pretrained.
4. **Sau mỗi VE:** validate 2 view (native + processed cố định), `val_loss = 0.5·CE_native + 0.5·CE_processed`,
   early stopping patience 2. Lưu `checkpoint_veN.pt`; `best.pt` = VE có val_loss thấp nhất.
   Mỗi checkpoint được nạp lại bằng `weights_only=True` và kiểm tra đúng định dạng evaluator cũ.

## Thời gian (ước lượng từ tốc độ GSD-aug-Tiny/B-Free đã đo, chưa đo trực tiếp)
- 1 VE ≈ 4 giờ train + ≈ 0,5 giờ validate trên T4. Một phiên Kaggle 12 giờ ≈ audit + 2 VE.
- `time_budget_hours=11`: gần hết giờ, notebook **tự lưu `last.pt` và dừng sạch** (status `paused_time_budget`).

## Nếu bị dừng vì hết giờ (resume)
1. Mở notebook → **Add Input → Your Work/Notebook Output** → chọn output của lần chạy trước.
2. Giữ nguyên config, **Save & Run All** lại. Code tự tìm `run_state.json` cùng cấu hình, chép sang
   `/kaggle/working`, tiếp tục **đúng bước** đã dừng (kể cả giữa VE), không audit lại.
3. Khi `status = complete`, `last.pt` (trạng thái optimizer ~3,6 GB) được xóa cho Output gọn; giữ `best.pt`.

## Output (`/kaggle/working/gsd_bfree_samecat_origbg_v1/`)
`best.pt` · `checkpoint_ve1.pt` … · `history.json` · `train_log.csv` · `validation_predictions_veN.csv`
(xác suất real/fake + nhóm diện tích) · `audit_*.csv/json` · `split_summary.json` · `run_state.json` ·
`selected_checkpoint.json` · `completion.json` (khi xong).

## So sánh với GSD-aug-Tiny
Dùng 2 notebook đi kèm (cùng code evaluator cũ, chỉ thêm một cell nhận checkpoint B-Free):
`test_gsd_bfree_samecat_origbg_tiny9_kaggle.ipynb` (Tiny 9 case, 7.000 ảnh/case) và
`test_gsd_bfree_samecat_origbg_bfree_valid_kaggle.ipynb` (B-Free valid 10.304 ID × 6 nhánh).
`checkpoint_ve1.pt` (25.200 ảnh) = so sánh cùng lượng dữ liệu với Tiny (câu hỏi P1).

In [ ]:
%pip install -q "transformers==4.46.3" "huggingface_hub>=0.23.2,<1.0" "pandas<3" "scikit-learn<1.9" "pillow<12" matplotlib tqdm

## Runtime (tự chứa: runtime GSD-aug-Tiny + lớp dữ liệu/train B-Free)

In [ ]:
"""Paired full-CLIP/GSD experiment. Self-contained Kaggle runtime.

Paper v2 Fig.3: GSD is BEFORE each of the last four transformer blocks.
No fake-only codec transformations, test tuning, or automatic batch-size changes.
"""
from __future__ import annotations

import copy
import gc
import hashlib
import io
import json
import math
import platform
import random
from pathlib import Path

import numpy as np
import pandas as pd
import PIL
from PIL import Image, ImageFilter, ImageOps
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, confusion_matrix, average_precision_score
from transformers import CLIPVisionModel, CLIPVisionConfig
from tqdm.auto import tqdm


DEFAULT_CONFIG = dict(
    dataset_root=None, input_root='/kaggle/input', output_root='/kaggle/working/clip_gsd_tiny',
    model_id='openai/clip-vit-large-patch14', local_files_only=False,
    seeds=[42], models=['clip_gsd'], epochs=50,
    early_stopping=True, patience=3, min_delta=0.001,
    train_batch=128, test_batch=512, micro_batch=2, anchor_micro_batch=8,
    lr=1e-6, head_lr=1e-6, modulation_lr=1e-6, weight_decay=0.01,
    betas=[0.9, 0.999], eps=1e-8, k=64, lambda_max=1., a_init=2., b_init=1.,
    validation_fraction=0.1, num_workers=2, gradient_checkpointing=True,
    precision='fp16', svd_driver='gesvdj', resume=True,
    jpeg_probability=0.5, jpeg_train_quality=[75, 100], jpeg_subsampling=0,
    blur_probability=0.5, blur_sigma=[0., 2.], horizontal_flip_probability=0.5,
    jpeg_eval_qualities=[96], size_control=256, threshold=0.5,
    pooling='patch_mean_post_ln', source_generator='sd15', strict_eight_generators=False,
    expected_generators=['sd15', 'adm', 'biggan', 'glide', 'midjourney', 'wukong', 'vqdm'],
    smoke=False, smoke_train_per_class=64, smoke_eval_per_class=4,
)


def dump(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False, default=str), encoding='utf-8')


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str).encode()).hexdigest()


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def canonical_generator(name):
    n = ''.join(c for c in name.lower() if c.isalnum())
    aliases = {'sd14': ('sd14', 'sdv14', 'sdv4', 'stablediffusionv14', 'stablediffusion14'),
               'sd15': ('sd15', 'sdv15', 'sdv5', 'stablediffusionv15', 'stablediffusion15'),
               'adm': ('adm', 'guided'), 'biggan': ('biggan',), 'glide': ('glide',),
               'midjourney': ('midjourney',), 'wukong': ('wukong',), 'vqdm': ('vqdm',)}
    matches = [key for key, values in aliases.items() if any(v in n for v in values)]
    return matches[0] if len(matches) == 1 else None


def find_root(cfg):
    if cfg['dataset_root']:
        return Path(cfg['dataset_root'])
    roots = set()
    for p in Path(cfg['input_root']).rglob('*'):
        if p.is_dir() and canonical_generator(p.name):
            if any((p / s).is_dir() for s in ('train', 'training')):
                roots.add(p.parent)
    if len(roots) != 1:
        raise ValueError(f'Set dataset_root explicitly. Candidate roots: {sorted(map(str, roots))}')
    return roots.pop()


def scan_dataset(cfg, out):
    root = find_root(cfg)
    rows, errors = [], []
    extensions = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
    for g in sorted(root.iterdir()):
        gen = canonical_generator(g.name)
        if not g.is_dir() or gen is None:
            continue
        for split, names in [('train', ['train', 'training']), ('test', ['val', 'valid', 'validation', 'test'])]:
            available = [g / s for s in names if (g / s).is_dir()]
            if len(available) != 1:
                raise ValueError(f'{g}: expected exactly one {split} split, found {available}')
            labels = {0: [], 1: []}
            mapping = {'nature': 0, 'real': 0, '0_real': 0, 'ai': 1, 'fake': 1, '1_fake': 1}
            for folder in available[0].iterdir():
                if folder.is_dir() and folder.name.lower() in mapping:
                    labels[mapping[folder.name.lower()]].append(folder)
            if any(len(v) != 1 for v in labels.values()):
                raise ValueError(f'Unexpected label folders: {available[0]} {labels}')
            for label, folders in labels.items():
                for path in sorted(folders[0].rglob('*')):
                    if path.suffix.lower() not in extensions or not path.is_file():
                        continue
                    # Audit source-train only: other generators' train images are not used.
                    if split == 'train' and gen not in cfg.get('train_generators', [cfg['source_generator']]):
                        continue
                    try:
                        raw = path.read_bytes()
                        with Image.open(io.BytesIO(raw)) as im:
                            fmt, w, h = im.format, *im.size
                            qtable = im.quantization if hasattr(im, 'quantization') else None
                            rgb = im.convert('RGB')
                            pixel_hash = hashlib.sha256(f'{w}x{h}:'.encode() + rgb.tobytes()).hexdigest()
                        rel = path.relative_to(root).as_posix()
                        rows.append(dict(sample_id=hashlib.sha256(rel.encode()).hexdigest(),
                                         path=str(path), relative_path=rel, generator=gen, split=split,
                                         label=label, width=w, height=h, format=fmt,
                                         pixel_hash=pixel_hash, file_hash=hashlib.sha256(raw).hexdigest(),
                                         qtable_hash=digest(qtable) if qtable else '', file_bytes=len(raw)))
                        if cfg.get('scan_log_every') and len(rows) % cfg['scan_log_every'] == 0:
                            print(f'[scan] {len(rows):,} images; {gen}/{split}; invalid={len(errors)}', flush=True)
                    except Exception as e:
                        errors.append(dict(path=str(path), error=repr(e)))
    df = pd.DataFrame(rows)
    pd.DataFrame(errors).to_csv(out / 'invalid_images.csv', index=False)
    if df.empty:
        raise ValueError('No usable images')
    if (df.groupby('pixel_hash').label.nunique() > 1).any():
        raise ValueError('Identical decoded pixels have conflicting labels; inspect dataset')
    df.to_csv(out / 'image_audit.csv', index=False)
    df.groupby(['split', 'generator', 'label', 'format']).size().rename('count').to_csv(out / 'format_counts.csv')
    found = set(df.loc[df.split == 'test', 'generator'])
    expected = set(cfg.get('expected_generators') or [])
    if not cfg['smoke'] and expected and found != expected:
        raise ValueError(f'Test generators mismatch: missing={sorted(expected-found)}, '
                         f'unexpected={sorted(found-expected)}, found={sorted(found)}. '
                         'Inspect image_audit.csv and invalid_images.csv.')
    if cfg['strict_eight_generators'] and len(found) != 8:
        raise ValueError(f'Expected eight test generators; found {sorted(found)}')
    for source_gen in cfg.get('train_generators', [cfg['source_generator']]):
        source = df[(df.split == 'train') & (df.generator == source_gen)]
        if set(source.label) != {0, 1}:
            raise ValueError(f"Source {source_gen!r} needs real and fake training images. "
                             'yangsangtai/tiny-genimage excludes SD14; use source_generator=sd15.')
    return df


def balanced(frame, seed, limit=None):
    n = int(frame.label.value_counts().reindex([0, 1], fill_value=0).min())
    n = min(n, limit) if limit else n
    if n == 0:
        raise ValueError('Every cohort must contain real and fake')
    return pd.concat([frame[frame.label == y].sample(n=n, random_state=seed + y) for y in (0, 1)])


def make_splits(df, cfg, seed, out):
    source = df[(df.split == 'train') & (df.generator == cfg['source_generator'])].drop_duplicates('pixel_hash')
    if cfg['smoke']:
        source = balanced(source, seed, cfg['smoke_train_per_class'])
    source = balanced(source, seed)
    tr, va = train_test_split(source, test_size=cfg['validation_fraction'], stratify=source.label, random_state=seed)
    used = set(source.pixel_hash)
    tests = df[df.split == 'test'].copy()
    removed = tests[tests.pixel_hash.isin(used)]
    removed.to_csv(out / 'excluded_test_overlap.csv', index=False)
    tests = tests[~tests.pixel_hash.isin(used)]
    tests = tests.drop_duplicates(['generator', 'pixel_hash'])
    tests = pd.concat([balanced(g, seed, cfg['smoke_eval_per_class'] if cfg['smoke'] else None)
                       for _, g in tests.groupby('generator')])
    # Label-blind order shared by all models and processing cases; never sort batches by label.
    def ordered(frame):
        frame = frame.copy()
        frame['_order'] = frame.sample_id.map(lambda s: digest([seed, s]))
        return frame.sort_values('_order').drop(columns='_order').reset_index(drop=True)
    tr, va, tests = ordered(tr), ordered(va), ordered(tests)
    for name, frame in [('train', tr), ('validation', va), ('test', tests)]:
        frame.to_csv(out / f'{name}_manifest.csv', index=False)
    dump(out / 'split_summary.json', dict(train=len(tr), validation=len(va), test=len(tests),
         test_overlap_removed=len(removed), test_repeated_real_pixels=int(tests[tests.label == 0].pixel_hash.duplicated().sum()),
         train_hash=digest(tr.sample_id.tolist()), validation_hash=digest(va.sample_id.tolist()),
         test_hash=digest(tests.sample_id.tolist()), source_generator=cfg['source_generator'],
         selection=f"source {cfg['source_generator']} train only; no test selection"))
    return tr, va, tests


def jpeg(image, quality, subsampling):
    buf = io.BytesIO()
    image.save(buf, format='JPEG', quality=int(quality), subsampling=subsampling,
               optimize=False, progressive=False)
    buf.seek(0)
    with Image.open(buf) as decoded:
        return decoded.convert('RGB')


def processing_cases(cfg):
    return ['native', 'size'] + [c for q in cfg['jpeg_eval_qualities'] for c in (f'jpeg{q}', f'size_jpeg{q}')]


def process(image, case, cfg):
    # Size stress-test includes square warp; explicitly NOT claimed to remove all size/history bias.
    if case.startswith('size'):
        image = image.resize((cfg['size_control'], cfg['size_control']), Image.Resampling.BICUBIC)
    if 'jpeg' in case:
        quality = int(case.split('jpeg')[-1])
        image = jpeg(image, quality, cfg['jpeg_subsampling'])
    return image


class Images(Dataset):
    def __init__(self, frame, cfg, seed, train=False, epoch=0, case='native'):
        self.rows = frame.to_dict('records')
        self.cfg, self.seed, self.train, self.epoch, self.case = cfg, seed, train, epoch, case

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r, c = self.rows[i], self.cfg
        with Image.open(r['path']) as im:
            image = im.convert('RGB')
        if self.train:
            rng = random.Random(int(digest([self.seed, self.epoch, r['sample_id']])[:16], 16))
            if rng.random() < c['horizontal_flip_probability']:
                image = ImageOps.mirror(image)
            if rng.random() < c['blur_probability']:
                image = image.filter(ImageFilter.GaussianBlur(rng.uniform(*c['blur_sigma'])))
            if rng.random() < c['jpeg_probability']:
                image = jpeg(image, rng.randint(*c['jpeg_train_quality']), c['jpeg_subsampling'])
        else:
            image = process(image, self.case, c)
        # Explicit standard OpenAI CLIP: shorter-side 224 bicubic, center crop, RGB mean/std.
        w, h = image.size
        image = image.resize((224 if w <= h else int(224 * w / h),
                              224 if h <= w else int(224 * h / w)), Image.Resampling.BICUBIC)
        w, h = image.size
        left, top = int(round((w - 224) / 2)), int(round((h - 224) / 2))
        image = image.crop((left, top, left + 224, top + 224))
        x = torch.from_numpy(np.asarray(image, dtype=np.float32).copy()).permute(2, 0, 1) / 255.
        x = (x - torch.tensor([.48145466, .4578275, .40821073])[:, None, None]) / torch.tensor([.26862954, .26130258, .27577711])[:, None, None]
        return x, int(r['label']), i


def loader(frame, cfg, seed, train=False, epoch=0, case='native'):
    ds = Images(frame, cfg, seed, train, epoch, case)
    return DataLoader(ds, batch_size=cfg['train_batch'] if train else cfg['test_batch'],
                      shuffle=train, generator=torch.Generator().manual_seed(seed + epoch),
                      num_workers=cfg['num_workers'], pin_memory=torch.cuda.is_available(),
                      drop_last=train)


class SemanticSuppression(nn.Module):
    def __init__(self, a=2., b=1., maximum=1.):
        super().__init__()
        self.a, self.b = nn.Parameter(torch.tensor(float(a))), nn.Parameter(torch.tensor(float(b)))
        self.maximum = maximum

    def forward(self, features, basis):
        # FP32 projection/occupancy; do not detach occupancy (Eq.10 differentiated end-to-end).
        with torch.autocast(features.device.type, enabled=False):
            f = features.float()
            p = (f @ basis.float()) @ basis.float().T
            r = p.square().sum((1, 2)) / f.square().sum((1, 2)).clamp_min(1e-12)
            lam = self.maximum * torch.sigmoid(self.a * r + self.b)
            return (f - lam[:, None, None] * p).to(features.dtype)


def top_basis(matrix, k, driver=None):
    with torch.autocast(matrix.device.type, enabled=False):
        options = {'driver': driver} if matrix.is_cuda and driver else {}
        _, _, vh = torch.linalg.svd(matrix.float(), full_matrices=False, **options)
        if k > min(matrix.shape):
            raise ValueError('k exceeds available dimensions')
        return vh[:k].T.contiguous().detach()


class Detector(nn.Module):
    def __init__(self, vision, cfg, gsd):
        super().__init__()
        self.vision = vision
        self.cfg = cfg
        self.selected = tuple(range(len(vision.encoder.layers) - 4, len(vision.encoder.layers)))
        d = vision.config.hidden_size
        self.head = nn.Linear(d, 2)
        self.gsd = nn.ModuleDict({str(l): SemanticSuppression(cfg['a_init'], cfg['b_init'], cfg['lambda_max'])
                                  for l in self.selected}) if gsd else nn.ModuleDict()

    def forward(self, x, bases=None):
        h = self.vision.pre_layrnorm(self.vision.embeddings(x))
        for l, block in enumerate(self.vision.encoder.layers):
            if str(l) in self.gsd:
                h = self.gsd[str(l)](h, bases[l])
            def apply(t, layer=block):
                return layer(t, attention_mask=None, causal_attention_mask=None)[0]
            h = checkpoint(apply, h, use_reentrant=False) if self.training and self.cfg['gradient_checkpointing'] else apply(h)
        h = self.vision.post_layernorm(h)
        if self.cfg['pooling'] == 'patch_mean_post_ln':
            h = h[:, 1:].mean(1)
        elif self.cfg['pooling'] == 'all_mean_post_ln':
            h = h.mean(1)
        else:
            raise ValueError('Unsupported pooling')
        return self.head(h)


def amp(cfg, device):
    return torch.autocast(device.type, dtype=torch.float16 if cfg['precision'] == 'fp16' else torch.bfloat16,
                          enabled=device.type == 'cuda' and cfg['precision'] != 'fp32')


@torch.no_grad()
def estimate_bases(anchor, images, cfg, device, selected):
    # Anchor PRE-block states, matching Fig.3. Collect the ENTIRE logical batch on CPU.
    chunks = {l: [] for l in selected}
    for x in images.split(cfg['anchor_micro_batch']):
        with amp(cfg, device):
            h = anchor.pre_layrnorm(anchor.embeddings(x.to(device)))
            for l, layer in enumerate(anchor.encoder.layers):
                if l in chunks:
                    chunks[l].append(h[:, 1:].detach().cpu())
                h = layer(h, attention_mask=None, causal_attention_mask=None)[0]
    bases = {}
    for l in selected:
        m = torch.cat(chunks.pop(l), dim=0).flatten(0, 1).to(device=device, dtype=torch.float32)
        bases[l] = top_basis(m, cfg['k'], cfg['svd_driver'])
        del m
    return bases


def build_models(cfg, device, gsd):
    pretrained = CLIPVisionModel.from_pretrained(cfg['model_id'], local_files_only=cfg['local_files_only'], attn_implementation='eager')
    v = pretrained.vision_model
    assert (v.config.hidden_size, v.config.num_hidden_layers, v.config.patch_size, v.config.image_size) == (1024, 24, 14, 224), 'Expected CLIP ViT-L/14 224'
    model = Detector(v, cfg, gsd).to(device)
    anchor = copy.deepcopy(v).requires_grad_(False).eval().to(device) if gsd else None
    if anchor is not None and device.type == 'cuda' and cfg['precision'] != 'fp32':
        anchor.to(dtype=torch.float16 if cfg['precision'] == 'fp16' else torch.bfloat16)
    return model, anchor


@torch.no_grad()
def predict(model, anchor, frame, cfg, seed, device, case):
    model.eval()
    result = np.full(len(frame), np.nan)
    batches = np.full(len(frame), -1, dtype=np.int64)
    for batch_id, (images, _, indices) in enumerate(tqdm(loader(frame, cfg, seed, case=case), desc=case)):
        bases = estimate_bases(anchor, images, cfg, device, model.selected) if anchor is not None else None
        for start in range(0, len(images), cfg['micro_batch']):
            ix = indices[start:start + cfg['micro_batch']].numpy()
            with amp(cfg, device):
                logits = model(images[start:start + cfg['micro_batch']].to(device), bases)
            result[ix] = logits.float().softmax(-1)[:, 1].cpu().numpy()
            batches[ix] = batch_id
    if not np.isfinite(result).all():
        raise ValueError('Nonfinite or missing predictions')
    return result, batches


def metric(y, p, threshold):
    pred = p >= threshold
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(n=len(y), auc=float(roc_auc_score(y, p)), ap=float(average_precision_score(y, p)),
                balanced_accuracy=float(balanced_accuracy_score(y, pred)), accuracy=float(np.mean(y == pred)),
                fake_recall=float(tp / (tp + fn)), real_recall=float(tn / (tn + fp)),
                tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))


class EarlyStopping:
    """Track significant validation-loss improvements, separately from best checkpoint."""
    def __init__(self, enabled=True, patience=3, min_delta=0.001, state=None):
        if patience < 1 or min_delta < 0:
            raise ValueError('patience must be >=1; min_delta must be >=0')
        self.enabled, self.patience, self.min_delta = enabled, patience, min_delta
        self.state = dict(reference_loss=float('inf'), bad_epochs=0, stopped=False)
        if state is not None:
            self.state.update(state)

    def update(self, loss):
        if not math.isfinite(loss):
            raise FloatingPointError('Nonfinite validation loss')
        if loss < self.state['reference_loss'] - self.min_delta:
            self.state.update(reference_loss=loss, bad_epochs=0)
        else:
            self.state['bad_epochs'] += 1
        self.state['stopped'] = bool(self.enabled and self.state['bad_epochs'] >= self.patience)
        return self.state['stopped']


def train_model(model, anchor, train, val, cfg, seed, device, folder, signature):
    groups = [dict(params=model.vision.parameters(), lr=cfg['lr']),
              dict(params=model.head.parameters(), lr=cfg['head_lr'])]
    if len(model.gsd):
        groups.append(dict(params=model.gsd.parameters(), lr=cfg['modulation_lr']))
    optimizer = torch.optim.AdamW(groups, betas=tuple(cfg['betas']), eps=cfg['eps'], weight_decay=cfg['weight_decay'])
    scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda' and cfg['precision'] == 'fp16')
    start, best, history = 0, float('inf'), []
    stopper = EarlyStopping(cfg['early_stopping'], cfg['patience'], cfg['min_delta'])
    latest = folder / 'last.pt'
    if cfg['resume'] and latest.exists():
        ck = torch.load(latest, map_location='cpu', weights_only=False)
        if ck['signature'] != signature:
            raise ValueError('Resume protocol/config/manifest changed; use a fresh output_root')
        model.load_state_dict(ck['model'])
        optimizer.load_state_dict(ck['optimizer'])
        scaler.load_state_dict(ck['scaler'])
        start, best, history = ck['epoch'] + 1, ck['best'], ck['history']
        if 'early_stopping' in ck:
            stopper.state.update(ck['early_stopping'])
        del ck
    if len(train) < cfg['train_batch']:
        raise ValueError('Not enough source-training images for a full logical batch')
    for epoch in range(start, cfg['epochs']):
        if stopper.state['stopped']:
            print('Resuming an early-stopped run: restore best checkpoint and evaluate.')
            break
        model.train()
        running, count, skipped = 0., 0, 0
        for images, labels, _ in tqdm(loader(train, cfg, seed, True, epoch), desc=f'epoch {epoch+1}'):
            bases = estimate_bases(anchor, images, cfg, device, model.selected) if anchor is not None else None
            optimizer.zero_grad(set_to_none=True)
            for a in range(0, len(images), cfg['micro_batch']):
                x, y = images[a:a + cfg['micro_batch']].to(device), labels[a:a + cfg['micro_batch']].to(device)
                with amp(cfg, device):
                    logits = model(x, bases)
                    loss = F.cross_entropy(logits.float(), y, reduction='sum') / len(images)
                if not torch.isfinite(loss):
                    raise FloatingPointError('Nonfinite training loss')
                scaler.scale(loss).backward()
                running += float(loss.detach()) * len(images)
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            skipped += int(scaler.get_scale() < old_scale)
            count += len(images)
        p, _ = predict(model, anchor, val, cfg, seed, device, 'native')
        vloss = float(F.binary_cross_entropy(torch.from_numpy(p).double().clamp(1e-7, 1-1e-7),
                                             torch.tensor(val.label.to_numpy(), dtype=torch.float64)))
        entry = dict(epoch=epoch+1, train_loss=running/count, val_loss=vloss, amp_skipped_steps=skipped,
                     **metric(val.label.to_numpy(), p, cfg['threshold']))
        entry['modulation'] = {l: dict(a=float(g.a), b=float(g.b)) for l, g in model.gsd.items()}
        should_stop = stopper.update(vloss)
        entry['early_stopping'] = dict(stopper.state)
        history.append(entry)
        print(entry)
        if vloss < best:
            best = vloss
            torch.save(dict(model=model.state_dict(), epoch=epoch, signature=signature, config=cfg), folder / 'best.pt')
        torch.save(dict(model=model.state_dict(), optimizer=optimizer.state_dict(), scaler=scaler.state_dict(),
                        epoch=epoch, best=best, history=history, signature=signature,
                        early_stopping=dict(stopper.state)), latest)
        dump(folder / 'history.json', history)
        dump(folder / 'training_status.json', dict(epoch=epoch+1, max_epochs=cfg['epochs'],
             early_stopping=dict(stopper.state), patience=cfg['patience'], min_delta=cfg['min_delta']))
        if should_stop:
            print(f"Early stopping at epoch {epoch+1}: no significant val-loss improvement for {cfg['patience']} epochs.")
            break
    ck = torch.load(folder / 'best.pt', map_location='cpu', weights_only=False)
    model.load_state_dict(ck['model'])
    dump(folder / 'selected_checkpoint.json', dict(epoch=ck['epoch']+1, criterion='minimum source-validation cross entropy', signature=signature))


def summarize(predictions, cfg, out):
    metrics, shifts = [], []
    for (seed, model, case), frame in predictions.groupby(['seed', 'model', 'case']):
        for gen, g in [('pooled', frame)] + list(frame.groupby('generator')):
            metrics.append(dict(seed=seed, model=model, case=case, generator=gen,
                                seen=(gen == cfg['source_generator']), **metric(g.label.to_numpy(), g.prob_fake.to_numpy(), cfg['threshold'])))
    metric_df = pd.DataFrame(metrics)
    metric_df.to_csv(out / 'metrics.csv', index=False)
    per_gen = metric_df[metric_df.generator != 'pooled']
    for protocol, g in [('all', per_gen), ('unseen', per_gen[per_gen.generator != cfg['source_generator']])]:
        g.groupby(['seed', 'model', 'case'])[['auc', 'balanced_accuracy', 'fake_recall', 'real_recall', 'accuracy']].mean().to_csv(out / f'macro_{protocol}.csv')
    for (seed, model), group in predictions.groupby(['seed', 'model']):
        base = group[group.case == 'native'].set_index('sample_id')
        for case, changed in group[group.case != 'native'].groupby('case'):
            changed = changed.set_index('sample_id')
            if set(changed.index) != set(base.index):
                raise ValueError('Paired cohorts differ')
            changed = changed.loc[base.index]
            if not np.array_equal(base.batch_id, changed.batch_id):
                raise ValueError('GSD logical batch membership changed')
            delta = changed.prob_fake - base.prob_fake
            paired = base[['generator', 'label']].copy()
            paired['delta'] = delta
            paired['abs_delta'] = delta.abs()
            paired['flip'] = (changed.prob_fake >= cfg['threshold']) != (base.prob_fake >= cfg['threshold'])
            paired['native_prob'] = base.prob_fake
            paired['changed_prob'] = changed.prob_fake
            paired.reset_index().to_csv(out / f'paired_seed{seed}_{model}_{case}.csv', index=False)
            for (gen, label), g in paired.groupby(['generator', 'label']):
                shifts.append(dict(seed=seed, model=model, case=case, generator=gen, label=label,
                                   mean_delta=g.delta.mean(), mean_abs_delta=g.abs_delta.mean(), flip_rate=g.flip.mean(), n=len(g)))
    pd.DataFrame(shifts).to_csv(out / 'paired_summary.csv', index=False)
    comparison = per_gen.pivot(index=['seed', 'case', 'generator'], columns='model', values=['auc', 'balanced_accuracy', 'fake_recall'])
    comparison.to_csv(out / 'model_comparison.csv')
    if {'clip_ft', 'clip_gsd'} <= set(predictions.model):
        for name in ('auc', 'balanced_accuracy', 'fake_recall'):
            comparison[(name, 'gsd_minus_ft')] = comparison[(name, 'clip_gsd')] - comparison[(name, 'clip_ft')]
        comparison.to_csv(out / 'model_comparison.csv')
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10, 5))
    plot = per_gen[per_gen.generator != cfg['source_generator']].groupby(['model', 'case']).balanced_accuracy.mean().unstack(0)
    plot.plot.bar(ax=ax)
    ax.set_ylabel('Macro balanced accuracy (unseen generators)')
    ax.set_ylim(0, 1)
    fig.tight_layout()
    fig.savefig(out / 'unseen_balanced_accuracy.png', dpi=160)
    plt.close(fig)
    return metric_df


def run(config=None):
    cfg = {**DEFAULT_CONFIG, **(config or {})}
    if cfg['smoke']:
        cfg.update(epochs=1, train_batch=8, test_batch=8, k=4, strict_eight_generators=False)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if device.type != 'cuda':
        raise RuntimeError('Use Kaggle GPU; run local unit tests for CPU validation')
    if cfg['precision'] == 'bf16' and not torch.cuda.is_bf16_supported():
        raise ValueError('This GPU does not support BF16; use fp16 on T4/P100')
    if min(cfg['epochs'], cfg['micro_batch'], cfg['anchor_micro_batch'], cfg['train_batch'], cfg['test_batch']) < 1:
        raise ValueError('Batch sizes and epoch count must be positive')
    out = Path(cfg['output_root'])
    out.mkdir(parents=True, exist_ok=True)
    dump(out / 'config.json', cfg)
    import transformers
    dump(out / 'environment.json', dict(torch=torch.__version__, transformers=transformers.__version__,
         pillow=PIL.__version__, python=platform.python_version(), gpu=torch.cuda.get_device_name(0),
         use_devices=1, paper='https://arxiv.org/html/2603.09242v2', torch_threads=torch.get_num_threads()))
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    df = scan_dataset(cfg, out)
    predictions = []
    for seed in cfg['seeds']:
        split_out = out / f'seed{seed}'
        split_out.mkdir(exist_ok=True)
        train, val, tests = make_splits(df, cfg, seed, split_out)
        cfg_hash = digest({**cfg, 'train_files': train.file_hash.tolist(), 'val_files': val.file_hash.tolist(),
                           'test_files': tests.file_hash.tolist()})
        for model_name in cfg['models']:
            if model_name not in ('clip_ft', 'clip_gsd'):
                raise ValueError(model_name)
            seed_all(seed)  # Same backbone AND randomly initialized classifier across the two models.
            folder = split_out / model_name
            folder.mkdir(exist_ok=True)
            model, anchor = build_models(cfg, device, model_name == 'clip_gsd')
            dump(folder / 'architecture.json', dict(backbone=str(model.vision.config),
                 trainable_parameters=sum(p.numel() for p in model.parameters() if p.requires_grad),
                 gsd_pre_block_indices=list(model.selected) if anchor else [],
                 initialization_head_sha=digest(model.head.weight.detach().cpu().tolist()),
                 pooling=cfg['pooling'], batch_basis='entire logical batch, independent of microbatch'))
            train_model(model, anchor, train, val, cfg, seed, device, folder, digest([cfg_hash, model_name]))
            for case in processing_cases(cfg):
                p, batches = predict(model, anchor, tests, cfg, seed, device, case)
                frame = tests[['sample_id', 'generator', 'label', 'pixel_hash']].copy()
                frame = frame.assign(seed=seed, model=model_name, case=case, prob_fake=p, batch_id=batches)
                frame.to_csv(folder / f'predictions_{case}.csv', index=False)
                predictions.append(frame)
            del model, anchor
            gc.collect()
            torch.cuda.empty_cache()
    predictions = pd.concat(predictions, ignore_index=True)
    predictions.to_csv(out / 'predictions.csv', index=False)
    metrics = summarize(predictions, cfg, out)
    dump(out / 'completion.json', dict(status='complete', smoke=cfg['smoke'], config_hash=digest(cfg)))
    print(metrics[metrics.generator == 'pooled'].to_string(index=False))
    return metrics


"""Native-resolution, raw-only All7 training. Also appended to the base runtime in the notebook."""
if 'DEFAULT_CONFIG' not in globals():
    from pathlib import Path
    _base_file = Path(__file__).with_name('runtime.py')
    exec(compile(_base_file.read_text(encoding='utf-8'), str(_base_file), 'exec'), globals())

import time

GENS7 = ['sd15', 'adm', 'biggan', 'glide', 'midjourney', 'wukong', 'vqdm']
DEFAULT_CONFIG = dict(DEFAULT_CONFIG, output_root='/kaggle/working/gsd_raw_all7_native_v1',
    source_generator='all7', train_generators=GENS7, expected_generators=GENS7,
    train_batch=8, test_batch=8, micro_batch=1, anchor_micro_batch=1, num_workers=0,
    jpeg_probability=0., blur_probability=0., horizontal_flip_probability=0.,
    input_mode='native_rgb_edge_pad14', basis_method='gram_eigh', attention='sdpa',
    eval_cases=['native'], scan_log_every=500)

_base_scan = scan_dataset

def scan_dataset(cfg, out):
    print('[scan] Reading all seven train/val folders; RGB decoding and hashing. No resize.', flush=True)
    start = time.monotonic()
    frame = _base_scan(cfg, out)
    frame.groupby(['split','generator','label']).size().rename('n').to_csv(out/'dataset_counts.csv')
    print(f'[scan] {len(frame):,} images audited in {time.monotonic()-start:.1f}s', flush=True)
    return frame


def make_splits(df, cfg, seed, out):
    source = df[(df.split=='train') & df.generator.isin(cfg['train_generators'])].copy()
    # Assign each decoded-pixel group ONCE, even when it occurs in several generators.
    unique = source.drop_duplicates('pixel_hash')
    strata = unique.generator + ':' + unique.label.astype(str)
    a, b = train_test_split(unique, test_size=cfg['validation_fraction'],
                          stratify=strata, random_state=seed)
    def select(frame, limit):
        groups = [g.drop_duplicates('pixel_hash') for _,g in frame.groupby('generator',sort=True)]
        if set(frame.generator)!=set(cfg['train_generators']):
            raise ValueError('Every configured train generator must have a nonempty cohort')
        n = min(int(g.label.value_counts().reindex([0,1],fill_value=0).min()) for g in groups)
        if limit is not None:
            n = min(n, limit)
        if n < 1:
            raise ValueError('A generator lacks real/fake images after hash-group splitting')
        return pd.concat([balanced(g, seed, n) for g in groups],ignore_index=True)
    tr = select(source[source.pixel_hash.isin(a.pixel_hash)], cfg['smoke_train_per_class'] if cfg['smoke'] else None)
    va = select(source[source.pixel_hash.isin(b.pixel_hash)], cfg['smoke_eval_per_class'] if cfg['smoke'] else None)
    tests = df[df.split=='test'].copy()
    removed = tests[tests.pixel_hash.isin(source.pixel_hash)]
    removed.to_csv(out/'excluded_test_overlap.csv',index=False)
    tests = tests[~tests.pixel_hash.isin(source.pixel_hash)].drop_duplicates(['generator','pixel_hash'])
    tests = pd.concat([balanced(g,seed,cfg['smoke_eval_per_class'] if cfg['smoke'] else None)
                       for _,g in tests.groupby('generator',sort=True)],ignore_index=True)
    def ordered(f):
        return f.assign(_order=f.sample_id.map(lambda s:digest([seed,s]))).sort_values('_order').drop(columns='_order').reset_index(drop=True)
    tr,va,tests=map(ordered,(tr,va,tests))
    assert not set(tr.pixel_hash)&set(va.pixel_hash)
    assert not (set(tr.pixel_hash)|set(va.pixel_hash))&set(tests.pixel_hash)
    for name,frame in [('train',tr),('validation',va),('test',tests)]:
        frame.to_csv(out/f'{name}_manifest.csv',index=False)
        frame.groupby(['generator','label']).size().rename('n').to_csv(out/f'{name}_counts.csv')
    dump(out/'split_summary.json',dict(train=len(tr),validation=len(va),test=len(tests),
        test_overlap_removed=len(removed),selection='All7 raw; unique pixel groups split before per-generator/class balancing',
        train_hash=digest(tr.sample_id.tolist()),validation_hash=digest(va.sample_id.tolist()),test_hash=digest(tests.sample_id.tolist())))
    print(f'[split] train={len(tr):,}, validation={len(va):,}, test={len(tests):,}; all 7 seen',flush=True)
    return tr,va,tests


def processing_cases(cfg):
    return cfg['eval_cases']


def input_geometry(width, height, cfg):
    """Source-pixel crop coordinates and exact model dimensions, without resampling."""
    cap = cfg['crop_max_side'] if cfg['input_mode']=='center_crop_max_edge_pad14' else None
    cw = min(width, cap) if cap else width
    ch = min(height, cap) if cap else height
    left, top = (width-cw)//2, (height-ch)//2
    pw, ph = ((cw+13)//14)*14, ((ch+13)//14)*14
    return dict(crop_left=left,crop_top=top,crop_width=cw,crop_height=ch,
                input_width=pw,input_height=ph,grid_w=pw//14,grid_h=ph//14,
                tokens=(pw//14)*(ph//14)+1)


def protocol_id(cfg):
    if cfg['input_mode']=='center_crop_max_edge_pad14':
        return f"raw_all7_center_crop{cfg['crop_max_side']}_v1"
    return 'raw_all7_native_v1'


class Images(Dataset):
    def __init__(self,frame,cfg,seed,train=False,epoch=0,case='native'):
        self.rows=frame.to_dict('records'); self.cfg=cfg; self.case=case; self.train=train
        if train and case!='native':
            raise ValueError('Training must use native raw images')

    def __len__(self):
        return len(self.rows)

    def __getitem__(self,i):
        row=self.rows[i]
        with Image.open(row['path']) as im:
            image=im.convert('RGB')
        if not self.train:
            if self.case.startswith('fake_jpeg'):
                if int(row['label'])==1:
                    image=jpeg(image,int(self.case[len('fake_jpeg'):]),self.cfg['jpeg_subsampling'])
            elif self.case!='native':
                image=process(image,self.case,self.cfg)
        geom=input_geometry(*image.size,self.cfg)
        image=image.crop((geom['crop_left'],geom['crop_top'],
                          geom['crop_left']+geom['crop_width'],geom['crop_top']+geom['crop_height']))
        x=torch.from_numpy(np.asarray(image,dtype=np.float32).copy()).permute(2,0,1)/255.
        # Preserve pixels inside the selected crop. No resize or upscaling of smaller images.
        h,w=x.shape[-2:]
        x=F.pad(x.unsqueeze(0),(0,(-w)%14,0,(-h)%14),mode='replicate').squeeze(0)
        mean=x.new_tensor([.48145466,.4578275,.40821073])[:,None,None]
        std=x.new_tensor([.26862954,.26130258,.27577711])[:,None,None]
        return (x-mean)/std,int(row['label']),i


def native_collate(items):
    x,y,i=zip(*items)
    return list(x),torch.tensor(y,dtype=torch.long),torch.tensor(i,dtype=torch.long)


def loader(frame,cfg,seed,train=False,epoch=0,case='native'):
    return DataLoader(Images(frame,cfg,seed,train,epoch,case),
        batch_size=cfg['train_batch'] if train else cfg['test_batch'], shuffle=train,
        generator=torch.Generator().manual_seed(seed+epoch),num_workers=cfg['num_workers'],
        collate_fn=native_collate,pin_memory=torch.cuda.is_available(),drop_last=False)


def image_chunks(images,limit):
    # Keep membership and order of the mixed logical batch, regardless of shape or label.
    start=0
    while start<len(images):
        end=start+1
        while end<min(start+limit,len(images)) and images[end].shape==images[start].shape:
            end+=1
        yield start,torch.stack(images[start:end])
        start=end


class NativeDetector(Detector):
    def forward(self,x,bases=None):
        h=self.vision.pre_layrnorm(self.vision.embeddings(x,interpolate_pos_encoding=True))
        for l,block in enumerate(self.vision.encoder.layers):
            if str(l) in self.gsd:
                h=self.gsd[str(l)](h,bases[l])
            def apply(t,layer=block):
                return layer(t,attention_mask=None,causal_attention_mask=None)[0]
            h=checkpoint(apply,h,use_reentrant=False) if self.training and self.cfg['gradient_checkpointing'] else apply(h)
        return self.head(self.vision.post_layernorm(h)[:,1:].mean(1))


@torch.no_grad()
def estimate_bases(anchor,images,cfg,device,selected):
    # M.T @ M has the same principal right-singular subspace as uncentered M.
    # Stream ALL tokens: no token sampling and no BN x d concatenation on GPU.
    grams={l:torch.zeros(anchor.config.hidden_size,anchor.config.hidden_size,device=device,dtype=torch.float32) for l in selected}
    tokens=0
    for _,x in image_chunks(images,cfg['anchor_micro_batch']):
        with amp(cfg,device):
            h=anchor.pre_layrnorm(anchor.embeddings(x.to(device),interpolate_pos_encoding=True))
            tokens+=h.shape[0]*(h.shape[1]-1)
            for l,layer in enumerate(anchor.encoder.layers):
                if l in grams:
                    with torch.autocast(device.type,enabled=False):
                        m=h[:,1:].flatten(0,1).float()
                        grams[l].add_(m.T@m)
                h=layer(h,attention_mask=None,causal_attention_mask=None)[0]
    if cfg['k']>min(tokens,anchor.config.hidden_size):
        raise ValueError(f"k={cfg['k']} exceeds token/feature dimensions of this logical batch ({tokens})")
    bases={}
    with torch.autocast(device.type,enabled=False):
        for l,g in grams.items():
            _,v=torch.linalg.eigh((g+g.T)*(0.5/tokens))
            bases[l]=v[:,-cfg['k']:].contiguous().detach()
    return bases


def build_models(cfg,device,gsd):
    pretrained=CLIPVisionModel.from_pretrained(cfg['model_id'],local_files_only=cfg['local_files_only'],attn_implementation=cfg['attention'])
    v=pretrained.vision_model
    if (v.config.hidden_size,v.config.patch_size)!=(1024,14):
        raise ValueError('Native All7 protocol requires CLIP ViT-L/14')
    model=NativeDetector(v,cfg,gsd).to(device)
    anchor=copy.deepcopy(v).requires_grad_(False).eval().to(device) if gsd else None
    if anchor is not None and device.type=='cuda' and cfg['precision']!='fp32':
        anchor.to(dtype=torch.float16 if cfg['precision']=='fp16' else torch.bfloat16)
    return model,anchor


@torch.no_grad()
def predict(model,anchor,frame,cfg,seed,device,case):
    model.eval(); result=np.full(len(frame),np.nan); batches=np.full(len(frame),-1)
    for bid,(images,_,indices) in enumerate(loader(frame,cfg,seed,case=case)):
        bases=estimate_bases(anchor,images,cfg,device,model.selected) if anchor is not None else None
        for start,x in image_chunks(images,cfg['micro_batch']):
            with amp(cfg,device):
                logits=model(x.to(device),bases)
            ix=indices[start:start+len(x)].numpy()
            result[ix]=logits.float().softmax(-1)[:,1].cpu().numpy(); batches[ix]=bid
        if bid%25==0:
            print(f'[{case}] batch {bid+1}/{math.ceil(len(frame)/cfg["test_batch"])}',flush=True)
    if not np.isfinite(result).all():
        raise ValueError('Missing/nonfinite prediction')
    return result,batches


def save_checkpoint(path,state):
    temp=path.with_suffix('.tmp')
    torch.save(state,temp)
    temp.replace(path)


def train_model(model,anchor,train,val,cfg,seed,device,folder,signature):
    groups=[dict(params=model.vision.parameters(),lr=cfg['lr']),dict(params=model.head.parameters(),lr=cfg['head_lr'])]
    if len(model.gsd):
        groups.append(dict(params=model.gsd.parameters(),lr=cfg['modulation_lr']))
    optimizer=torch.optim.AdamW(groups,betas=tuple(cfg['betas']),eps=cfg['eps'],weight_decay=cfg['weight_decay'])
    scaler=torch.amp.GradScaler('cuda',enabled=device.type=='cuda' and cfg['precision']=='fp16')
    stopper=EarlyStopping(cfg['early_stopping'],cfg['patience'],cfg['min_delta'])
    latest=folder/'last.pt'; start=0; best=float('inf'); history=[]
    if cfg['resume'] and latest.exists():
        ck=torch.load(latest,map_location='cpu',weights_only=False)
        if ck['signature']!=signature:
            raise ValueError('Resume config/manifests changed; use a new output_root')
        model.load_state_dict(ck['model']); optimizer.load_state_dict(ck['optimizer']); scaler.load_state_dict(ck['scaler'])
        stopper.state.update(ck['early_stopping']); start=ck['epoch']+1; best=ck['best']; history=ck['history']
        del ck
        print(f'[resume] next epoch {start+1}, stopped={stopper.state["stopped"]}',flush=True)
    for epoch in range(start,cfg['epochs']):
        if stopper.state['stopped']:
            break
        model.train(); running=0.; count=0; skipped=0; tick=time.monotonic()
        for bid,(images,labels,_) in enumerate(loader(train,cfg,seed,True,epoch)):
            bases=estimate_bases(anchor,images,cfg,device,model.selected) if anchor is not None else None
            optimizer.zero_grad(set_to_none=True)
            for a,x in image_chunks(images,cfg['micro_batch']):
                with amp(cfg,device):
                    logits=model(x.to(device),bases)
                    loss=F.cross_entropy(logits.float(),labels[a:a+len(x)].to(device),reduction='sum')/len(images)
                if not torch.isfinite(loss):
                    raise FloatingPointError('Nonfinite loss')
                scaler.scale(loss).backward(); running+=float(loss.detach())*len(images)
            old=scaler.get_scale(); scaler.step(optimizer); scaler.update(); skipped+=int(scaler.get_scale()<old); count+=len(images)
            if bid%10==0:
                print(f'[epoch {epoch+1}] batch {bid+1}/{math.ceil(len(train)/cfg["train_batch"])}; loss={running/count:.5f}',flush=True)
        p,_=predict(model,anchor,val,cfg,seed,device,'native')
        vloss=float(F.binary_cross_entropy(torch.from_numpy(p).double().clamp(1e-7,1-1e-7),torch.tensor(val.label.to_numpy(),dtype=torch.float64)))
        stop=stopper.update(vloss)
        entry=dict(epoch=epoch+1,train_loss=running/count,val_loss=vloss,seconds=time.monotonic()-tick,
            amp_skipped_steps=skipped,train_images=count,**metric(val.label.to_numpy(),p,cfg['threshold']),
            modulation={l:dict(a=float(g.a.detach()),b=float(g.b.detach())) for l,g in model.gsd.items()},early_stopping=dict(stopper.state))
        history.append(entry); print(entry,flush=True)
        if vloss<best:
            best=vloss; save_checkpoint(folder/'best.pt',dict(model=model.state_dict(),epoch=epoch,signature=signature,config=cfg))
        save_checkpoint(latest,dict(model=model.state_dict(),optimizer=optimizer.state_dict(),scaler=scaler.state_dict(),
            epoch=epoch,best=best,history=history,signature=signature,config=cfg,early_stopping=dict(stopper.state)))
        dump(folder/'history.json',history)
        dump(folder/'training_status.json',dict(epoch=epoch+1,max_epochs=cfg['epochs'],early_stopping=dict(stopper.state)))
        if stop:
            print('[early stopping] Restore best checkpoint for evaluation.',flush=True)
            break
    ck=torch.load(folder/'best.pt',map_location='cpu',weights_only=False); model.load_state_dict(ck['model'])
    dump(folder/'selected_checkpoint.json',dict(epoch=ck['epoch']+1,criterion='minimum All7 raw validation cross entropy',signature=signature))


def summarize(predictions,cfg,out):
    rows=[]
    for (seed,model,case),f in predictions.groupby(['seed','model','case']):
        for gen,g in [('pooled',f),*f.groupby('generator')]:
            rows.append(dict(seed=seed,model=model,case=case,generator=gen,
                seen=gen in cfg['train_generators'],**metric(g.label.to_numpy(),g.prob_fake.to_numpy(),cfg['threshold'])))
    result=pd.DataFrame(rows); result.to_csv(out/'metrics.csv',index=False)
    result[result.generator!='pooled'].groupby(['seed','model','case'])[['auc','balanced_accuracy','fake_recall','real_recall','accuracy']].mean().to_csv(out/'macro_all.csv')
    return result


def validate_raw_config(cfg):
    if set(cfg['train_generators'])!=set(GENS7) or set(cfg['expected_generators'])!=set(GENS7):
        raise ValueError('This protocol requires exactly all seven Tiny generators')
    if any(cfg[k]!=0 for k in ('jpeg_probability','blur_probability','horizontal_flip_probability')):
        raise ValueError('Raw-only training forbids JPEG/blur/flip augmentation')
    if cfg['input_mode'] not in ('native_rgb_edge_pad14','center_crop_max_edge_pad14') or cfg['basis_method']!='gram_eigh':
        raise ValueError('Unsupported native-input protocol')
    if cfg['input_mode']=='center_crop_max_edge_pad14':
        cap=cfg.get('crop_max_side')
        if isinstance(cap,bool) or not isinstance(cap,int) or cap<14 or cap%14:
            raise ValueError('crop_max_side must be a positive integer multiple of 14')
    if cfg['attention']!='sdpa' or cfg['pooling']!='patch_mean_post_ln':
        raise ValueError('Native protocol requires SDPA and patch_mean_post_ln')
    if min(cfg[k] for k in ('train_batch','test_batch','micro_batch','anchor_micro_batch','epochs','k'))<1:
        raise ValueError('Positive batches, epochs and k required')
    if cfg['precision'] not in ('fp16','bf16','fp32'):
        raise ValueError('precision must be fp16, bf16 or fp32')
    if not cfg['eval_cases'] or 'native' not in cfg['eval_cases']:
        raise ValueError('Evaluation must include native')
    for case in cfg['eval_cases']:
        if case not in ('native','size') and not any(case==prefix+str(q) for prefix in ('jpeg','fake_jpeg','size_jpeg') for q in range(1,101)):
            raise ValueError(f'Unknown evaluation case: {case}')


def run(config=None):
    cfg={**DEFAULT_CONFIG,**(config or {})}; validate_raw_config(cfg)
    if cfg['smoke']:
        cfg.update(epochs=1,train_batch=2,test_batch=2,k=4)
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if device.type!='cuda':
        raise RuntimeError('Use Kaggle GPU; CPU unit tests cover native input separately')
    if cfg['precision']=='bf16' and not torch.cuda.is_bf16_supported():
        raise ValueError('GPU does not support bf16; use fp16 on T4')
    out=Path(cfg['output_root']); out.mkdir(parents=True,exist_ok=True)
    if (out/'config.json').exists():
        previous=json.loads((out/'config.json').read_text(encoding='utf-8'))
        if previous!=cfg:
            raise ValueError('Existing output config differs; choose a new output_root before running')
    dump(out/'config.json',cfg)
    dump(out/'environment.json',dict(torch=torch.__version__,pillow=PIL.__version__,python=platform.python_version(),
        gpu=torch.cuda.get_device_name(0),use_devices=1,protocol=protocol_id(cfg)))
    torch.backends.cuda.matmul.allow_tf32=False; torch.backends.cudnn.allow_tf32=False
    df=scan_dataset(cfg,out)
    census=df[['sample_id','split','generator','label','width','height']].copy()
    geometry=pd.DataFrame([input_geometry(int(w),int(h),cfg) for w,h in zip(census.width,census.height)])
    census=pd.concat([census.reset_index(drop=True),geometry],axis=1)
    census.to_csv(out/'token_census.csv',index=False)
    print(f'[{protocol_id(cfg)}] tokens min/median/max: {census.tokens.min()}/{census.tokens.median()}/{census.tokens.max()}. No pixel resampling.',flush=True)
    results=[]
    for seed in cfg['seeds']:
        split_out=out/f'seed{seed}'; split_out.mkdir(exist_ok=True)
        train,val,tests=make_splits(df,cfg,seed,split_out)
        signature=digest(dict(config=cfg,train=train.file_hash.tolist(),val=val.file_hash.tolist(),test=tests.file_hash.tolist(),protocol=protocol_id(cfg)))
        for name in cfg['models']:
            if name not in ('clip_ft','clip_gsd'):
                raise ValueError(name)
            seed_all(seed); folder=split_out/name; folder.mkdir(exist_ok=True)
            print(f'[model] Loading {name}; native position interpolation in BOTH branches.',flush=True)
            model,anchor=build_models(cfg,device,name=='clip_gsd')
            dump(folder/'architecture.json',dict(backbone=str(model.vision.config),input_mode=cfg['input_mode'],
                basis=cfg['basis_method'],attention=cfg['attention'],all_generators_seen=True,patch_size=14,
                protocol=protocol_id(cfg),crop_max_side=cfg.get('crop_max_side'),
                padding='right/bottom edge replication to multiple of 14; no resize',
                trainable_parameters=sum(p.numel() for p in model.parameters() if p.requires_grad)))
            try:
                train_model(model,anchor,train,val,cfg,seed,device,folder,signature)
                for case in processing_cases(cfg):
                    p,batches=predict(model,anchor,tests,cfg,seed,device,case)
                    f=tests[['sample_id','generator','label','pixel_hash','width','height']].assign(seed=seed,model=name,case=case,prob_fake=p,batch_id=batches)
                    f.to_csv(folder/f'predictions_{case}.csv',index=False); results.append(f)
            except torch.cuda.OutOfMemoryError as exc:
                raise RuntimeError('CUDA OOM under the configured input protocol. No automatic resize, skip or batch reduction. Use more VRAM or explicitly start a smaller-input/batch protocol; last.pt retains completed epochs.') from exc
            del model,anchor; gc.collect(); torch.cuda.empty_cache()
    predictions=pd.concat(results,ignore_index=True); predictions.to_csv(out/'predictions.csv',index=False)
    metrics=summarize(predictions,cfg,out)
    dump(out/'completion.json',dict(status='complete',smoke=cfg['smoke'],protocol=protocol_id(cfg),config_hash=digest(cfg)))
    print(metrics[metrics.generator=='pooled'].to_string(index=False),flush=True)
    return metrics


"""Trusted audit-manifest reuse. Loaded after runtime.py; never decode/hash image content.

The caller deliberately trusts the previous image audit and the same dataset version.
Filesystem stat checks cannot detect content changes that keep the same byte length.
"""
from pathlib import Path, PurePosixPath
import re

MANIFEST_PARTITIONS = ('train', 'validation', 'test')
MANIFEST_COLUMNS = {
    'sample_id', 'relative_path', 'generator', 'split', 'label', 'width', 'height',
    'file_hash', 'pixel_hash', 'file_bytes',
}


def find_manifest_root(cfg):
    def complete(root):
        return all((root / f'seed{seed}' / f'{part}_manifest.csv').is_file()
                   for seed in cfg['seeds'] for part in MANIFEST_PARTITIONS)

    if cfg['manifest_root']:
        root = Path(cfg['manifest_root']).resolve()
        if re.fullmatch(r'seed\d+', root.name):
            root = root.parent
        if not complete(root):
            raise FileNotFoundError(f'Missing saved train/validation/test manifests under {root}/seed<seed>. '
                                    'Attach the previous All7 output; no full-scan fallback is enabled.')
        return root
    search = Path(cfg['input_root'])
    roots = sorted({p.parent.parent.resolve() for p in search.rglob('train_manifest.csv')
                    if p.parent.name == f'seed{cfg["seeds"][0]}' and complete(p.parent.parent)})
    if len(roots) != 1:
        raise ValueError(f'Set manifest_root explicitly. Complete All7-manifest candidates: {[str(p) for p in roots]}. '
                         'Need seed42/train_manifest.csv, validation_manifest.csv and test_manifest.csv; '
                         'weights alone are not enough. No image scan was started.')
    return roots[0]


def validate_manifest_frame(frame, cfg, partition):
    missing = MANIFEST_COLUMNS - set(frame)
    if missing or frame.empty:
        raise ValueError(f'Invalid {partition} manifest: missing columns={sorted(missing)} or empty cohort')
    frame = frame.copy()
    if frame[list(MANIFEST_COLUMNS)].isna().any().any():
        raise ValueError(f'Null required metadata in {partition} manifest')
    if frame.sample_id.duplicated().any() or frame.relative_path.duplicated().any():
        raise ValueError(f'Duplicate IDs/paths in {partition} manifest')
    for column in ('sample_id', 'file_hash', 'pixel_hash'):
        if not frame[column].astype(str).str.fullmatch(r'[0-9a-f]{64}').all():
            raise ValueError(f'Invalid saved {column} in {partition} manifest')
    for column in ('label', 'width', 'height', 'file_bytes'):
        values = pd.to_numeric(frame[column], errors='coerce')
        if not np.isfinite(values).all() or (values != np.floor(values)).any():
            raise ValueError(f'Invalid integer {column} in {partition} manifest')
        frame[column] = values.astype('int64')
    if not frame.label.isin([0, 1]).all() or (frame[['width', 'height', 'file_bytes']] <= 0).any().any():
        raise ValueError(f'Invalid labels/dimensions/byte lengths in {partition} manifest')
    source_split = 'test' if partition == 'test' else 'train'
    if set(frame.split) != {source_split}:
        raise ValueError(f'{partition} manifest must originate only from source {source_split} folders')
    counts = frame.groupby(['generator', 'label']).size()
    expected = {(g, y) for g in cfg['train_generators'] for y in (0, 1)}
    if set(counts.index) != expected or counts.nunique() != 1:
        raise ValueError(f'{partition} manifest must contain balanced real/fake cohorts for all seven generators')
    if frame.duplicated(['generator', 'pixel_hash']).any():
        raise ValueError(f'Duplicate pixels within a generator in {partition} manifest')
    label_names = {'nature': 0, 'real': 0, '0_real': 0, 'ai': 1, 'fake': 1, '1_fake': 1}
    split_names = {'train': {'train', 'training'}, 'test': {'val', 'valid', 'validation', 'test'}}
    for row in frame.itertuples():
        rel = str(row.relative_path)
        parts = PurePosixPath(rel).parts
        if (not parts or len(parts) < 4 or PurePosixPath(rel).is_absolute() or
                '\\' in rel or ':' in rel or '..' in parts or rel != PurePosixPath(rel).as_posix()):
            raise ValueError(f'Unsafe relative image path: {rel}')
        if (canonical_generator(parts[0]) != row.generator or parts[1] not in split_names[source_split] or
                label_names.get(parts[2].lower()) != row.label):
            raise ValueError(f'Path/generator/split/label mismatch: {rel}')
        if row.sample_id != hashlib.sha256(rel.encode()).hexdigest():
            raise ValueError(f'Saved sample_id does not match relative path: {rel}')
    return frame


def read_manifest_cohorts(root, cfg, seed):
    folder = root / f'seed{seed}'
    summary_path = folder / 'split_summary.json'
    summary = json.loads(summary_path.read_text(encoding='utf-8')) if summary_path.is_file() else None
    frames, sources = {}, {}
    for partition in MANIFEST_PARTITIONS:
        path = folder / f'{partition}_manifest.csv'
        data = path.read_bytes()  # Hash only the small CSV, NOT any source image.
        frame = pd.read_csv(io.BytesIO(data), dtype={k: str for k in
            ('sample_id', 'relative_path', 'generator', 'split', 'file_hash', 'pixel_hash')})
        frame = validate_manifest_frame(frame, cfg, partition)
        if summary is not None:
            if (summary.get(partition) != len(frame) or
                    summary.get(partition + '_hash') != digest(frame.sample_id.tolist())):
                raise ValueError(f'{path}: ordered cohort does not match saved split_summary.json')
        frames[partition] = frame
        sources[partition] = dict(path=str(path), csv_sha256=hashlib.sha256(data).hexdigest(),
                                  count=len(frame), ordered_ids_hash=digest(frame.sample_id.tolist()))
    for a, b in (('train', 'validation'), ('train', 'test'), ('validation', 'test')):
        if set(frames[a].sample_id) & set(frames[b].sample_id):
            raise ValueError(f'Saved IDs overlap across {a}/{b}')
        if set(frames[a].pixel_hash) & set(frames[b].pixel_hash):
            raise ValueError(f'Saved pixel hashes overlap across {a}/{b}')
    joined = pd.concat(list(frames.values()), ignore_index=True)
    if (joined.groupby('pixel_hash').label.nunique() > 1).any():
        raise ValueError('Conflicting saved labels for identical pixel hashes')
    return frames, sources


def resolve_manifest_dataset(frame, cfg):
    first = frame.relative_path.iloc[0]
    if cfg['dataset_root']:
        candidates = [Path(cfg['dataset_root'])]
    else:
        search = Path(cfg['input_root'])
        candidates = [search / 'tiny-genimage', search / 'datasets/yangsangtai/tiny-genimage']
        generator_dir = PurePosixPath(first).parts[0]
        print('[manifest] Locating dataset root using directory metadata only.', flush=True)
        candidates.extend(p.parent for p in search.rglob(generator_dir) if p.is_dir())
    matches = sorted({p.resolve() for p in candidates if (p / first).is_file()})
    if len(matches) != 1:
        raise ValueError(f'Set dataset_root to the directory containing generator folders. Candidates: '
                         f'{[str(p) for p in matches]}. No RGB/hash scan was started.')
    root = matches[0]
    resolved = frame.copy()
    paths = []
    for index, row in enumerate(resolved.itertuples()):
        path = (root / row.relative_path).resolve()
        try:
            path.relative_to(root)
        except ValueError:
            raise ValueError(f'Image path escapes dataset root: {row.relative_path}')
        if not path.is_file() or path.stat().st_size != row.file_bytes:
            raise ValueError(f'Missing or changed file (path/size check): {path}. '
                             'Use the same dataset version as the saved audit.')
        paths.append(str(path))
        if (index + 1) % 5000 == 0:
            print(f'[manifest] {index + 1:,}/{len(frame):,} file paths/sizes checked; no image decode/hash.', flush=True)
    resolved['path'] = paths  # Ignore stale absolute /kaggle/input paths from the previous run.
    return resolved, root


def load_manifest_index(cfg, out):
    root = find_manifest_root(cfg)
    source_config_path = root / 'config.json'
    if source_config_path.is_file():
        source = json.loads(source_config_path.read_text(encoding='utf-8'))
        if source.get('smoke') and not cfg['smoke']:
            raise ValueError('Cannot use smoke-only source manifests for a full training run')
        if set(source.get('train_generators', [])) != set(cfg['train_generators']):
            raise ValueError('Source config is not an All7 experiment')
        if source.get('validation_fraction') != cfg['validation_fraction']:
            raise ValueError('validation_fraction differs from saved split; reused manifests are never re-split')
    all_frames, cohorts, sources = [], {}, {}
    for seed in cfg['seeds']:
        frames, provenance = read_manifest_cohorts(root, cfg, seed)
        all_frames.extend(frames.values())
        cohorts[str(seed)] = {part: f.sample_id.tolist() for part, f in frames.items()}
        sources[str(seed)] = provenance
    merged = pd.concat(all_frames, ignore_index=True)
    # A sample may occur in several seed experiments; metadata must agree.
    for column in sorted(MANIFEST_COLUMNS - {'sample_id'}):
        if (merged.groupby('sample_id')[column].nunique(dropna=False) > 1).any():
            raise ValueError(f'Conflicting saved {column} across seeds/cohorts')
    merged = merged.drop_duplicates('sample_id', keep='first').reset_index(drop=True)
    merged, dataset = resolve_manifest_dataset(merged, cfg)
    provenance = dict(mode='reuse_manifests', manifest_root=str(root), dataset_root=str(dataset),
                      sources=sources, unique_selected_images=len(merged),
                      checks=['schema', 'ordered_split_summary_if_present', 'saved_pixel_hash_leakage',
                              'label_and_split_path_consistency', 'file_exists', 'file_byte_length'],
                      skipped=['image_decode', 'image_content_hash', 'pixel_hash_recomputation',
                               'discovery_of_unlisted_or_new_images'],
                      trust='Saved audit metadata and unchanged dataset version are trusted. '
                            'Same-size content changes are NOT detected.')
    target = out / 'manifest_reuse.json'
    if target.exists() and json.loads(target.read_text(encoding='utf-8')) != provenance:
        raise ValueError('Manifest/input fingerprint changed; choose a new output_root before resume')
    dump(target, provenance)
    save_csv(out / 'reused_manifest_index.csv', merged)
    merged.attrs['reused_cohorts'] = cohorts
    merged.attrs['manifest_root'] = str(root)
    print(f'[manifest] Reused {len(merged):,} selected images. No RGB decoding or image hashing; no re-split.', flush=True)
    return merged


def reuse_saved_splits(df, cfg, seed, out):
    cohorts = df.attrs.get('reused_cohorts', {}).get(str(seed))
    if cohorts is None:
        raise ValueError('Missing validated saved-cohort index; refusing to re-split or scan implicitly')
    indexed = df.set_index('sample_id', drop=False)
    frames = []
    for partition in MANIFEST_PARTITIONS:
        frame = indexed.loc[cohorts[partition]].reset_index(drop=True)
        frame.attrs = {}
        if cfg['smoke']:
            limit = cfg['smoke_train_per_class'] if partition == 'train' else cfg['smoke_eval_per_class']
            frame = frame[frame.groupby(['generator', 'label']).cumcount() < limit].reset_index(drop=True)
        save_csv(out / f'{partition}_manifest.csv', frame)
        frame.groupby(['generator', 'label']).size().rename('n').to_csv(out / f'{partition}_counts.csv')
        frames.append(frame)
    tr, va, te = frames
    dump(out / 'split_summary.json', dict(train=len(tr), validation=len(va), test=len(te),
         train_hash=digest(tr.sample_id.tolist()), validation_hash=digest(va.sample_id.tolist()),
         test_hash=digest(te.sample_id.tolist()), selection='Reused ordered audited All7 cohorts; no re-split',
         manifest_root=df.attrs['manifest_root'], smoke_subset=bool(cfg['smoke'])))
    print(f'[split/reused] train={len(tr):,}, validation={len(va):,}, test={len(te):,}; original ID order retained.', flush=True)
    return tr, va, te


"""All7 crop560 + label-independent blur/JPEG + fixed two-view validation.

Append after runtime.py and runtime_raw_all7.py in the standalone notebook.
The raw-only runtimes/notebooks remain unchanged.
"""
if 'GENS7' not in globals():
    from pathlib import Path
    _raw_file = Path(__file__).with_name('runtime_raw_all7.py')
    exec(compile(_raw_file.read_text(encoding='utf-8'), str(_raw_file), 'exec'), globals())

if 'load_manifest_index' not in globals():
    _manifest_file = Path(__file__).with_name('manifest_reuse.py')
    exec(compile(_manifest_file.read_text(encoding='utf-8'), str(_manifest_file), 'exec'), globals())

_raw_images = Images
_raw_validate = validate_raw_config
_raw_splits = make_splits
_raw_run = run
_full_scan = scan_dataset

VALIDATION_MONITOR = 'mean_native_fixed_processed_cross_entropy'
DEFAULT_CONFIG = dict(
    DEFAULT_CONFIG,
    output_root='/kaggle/working/gsd_all7_crop560_blur_jpeg_manifest_v2',
    data_mode='reuse_manifests', manifest_root=None,
    input_mode='center_crop_max_edge_pad14', crop_max_side=560,
    jpeg_probability=0.5, jpeg_train_quality=[75, 100], jpeg_subsampling=0,
    blur_probability=0.5, blur_sigma=[0.0, 2.0], horizontal_flip_probability=0.0,
    validation_processing_seed=42, validation_monitor=VALIDATION_MONITOR,
)


def protocol_id(cfg):
    return f"all7_center_crop{cfg['crop_max_side']}_blur_jpeg_fixedval_v2"


def validate_raw_config(cfg):
    # Reuse geometry, batch, evaluation-case and All7 safeguards, not the raw-only ban.
    _raw_validate(dict(cfg, jpeg_probability=0.0, blur_probability=0.0,
                       horizontal_flip_probability=0.0))
    if cfg['input_mode'] != 'center_crop_max_edge_pad14':
        raise ValueError('Blur/JPEG protocol requires the capped center-crop input mode')
    if cfg['horizontal_flip_probability'] != 0:
        raise ValueError('Only Gaussian blur and JPEG are enabled; flip must remain zero')
    for name in ('jpeg_probability', 'blur_probability'):
        value = cfg[name]
        if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value) or not 0 <= value <= 1:
            raise ValueError(f'{name} must be a finite probability in [0, 1]')
    quality = cfg['jpeg_train_quality']
    if (not isinstance(quality, (list, tuple)) or len(quality) != 2 or
            any(isinstance(v, bool) or not isinstance(v, int) for v in quality) or
            not 1 <= quality[0] <= quality[1] <= 100):
        raise ValueError('jpeg_train_quality must be two ordered integers in [1, 100]')
    sigma = cfg['blur_sigma']
    if (not isinstance(sigma, (list, tuple)) or len(sigma) != 2 or
            any(isinstance(v, bool) or not isinstance(v, (int, float)) or not math.isfinite(v) for v in sigma) or
            not 0 <= sigma[0] <= sigma[1]):
        raise ValueError('blur_sigma must be two finite ordered nonnegative numbers')
    if isinstance(cfg['jpeg_subsampling'], bool) or cfg['jpeg_subsampling'] != 0:
        raise ValueError('This controlled protocol fixes JPEG subsampling=0 (4:4:4)')
    if isinstance(cfg['validation_processing_seed'], bool) or not isinstance(cfg['validation_processing_seed'], int):
        raise ValueError('validation_processing_seed must be an integer')
    if cfg['validation_monitor'] != VALIDATION_MONITOR:
        raise ValueError('Early stopping must monitor equal-weight native/processed validation CE')
    if cfg['data_mode'] not in ('reuse_manifests', 'full_scan'):
        raise ValueError('data_mode must be reuse_manifests or explicit full_scan')
    if not cfg['seeds'] or len(set(cfg['seeds'])) != len(cfg['seeds']):
        raise ValueError('Need a nonempty list of distinct run seeds')


def scan_dataset(cfg, out):
    if cfg['data_mode'] == 'reuse_manifests':
        return load_manifest_index(cfg, out)
    return _full_scan(cfg, out)


def processing_parameters(sample_id, cfg, seed, train=False, epoch=0):
    """Local RNG: independent of labels, row order, workers and global random state.

    Validation deliberately ignores epoch; training changes its draws each epoch.
    The two Bernoulli gates are independent, including when an operation is disabled.
    """
    identity = ['blur_jpeg_v1', int(seed), str(sample_id)]
    identity += ['train', int(epoch)] if train else ['validation', cfg['validation_processing_seed']]
    rng = random.Random(int(digest(identity)[:16], 16))
    blur_applied = rng.random() < cfg['blur_probability']
    jpeg_applied = rng.random() < cfg['jpeg_probability']
    sigma = rng.uniform(*cfg['blur_sigma'])
    quality = rng.randint(*cfg['jpeg_train_quality'])
    return dict(blur_applied=blur_applied, blur_sigma=sigma if blur_applied else None,
                jpeg_applied=jpeg_applied, jpeg_quality=quality if jpeg_applied else None)


def apply_processing(image, parameters, cfg):
    # Operate on the decoded original, BEFORE crop/padding/normalization.
    if parameters['blur_applied']:
        image = image.filter(ImageFilter.GaussianBlur(parameters['blur_sigma']))
    if parameters['jpeg_applied']:
        image = jpeg(image, parameters['jpeg_quality'], cfg['jpeg_subsampling'])
    return image


class Images(_raw_images):
    def __init__(self, frame, cfg, seed, train=False, epoch=0, case='native'):
        super().__init__(frame, cfg, seed, train, epoch, case)
        self.seed, self.epoch = seed, epoch

    def __getitem__(self, i):
        if not self.train and self.case != 'processed':
            # Native/test cases remain byte-for-byte equivalent to the raw evaluator.
            return super().__getitem__(i)
        row = self.rows[i]
        with Image.open(row['path']) as im:
            image = im.convert('RGB')
        parameters = processing_parameters(row['sample_id'], self.cfg, self.seed,
                                           train=self.train, epoch=self.epoch)
        image = apply_processing(image, parameters, self.cfg)
        geom = input_geometry(*image.size, self.cfg)
        image = image.crop((geom['crop_left'], geom['crop_top'],
                            geom['crop_left'] + geom['crop_width'],
                            geom['crop_top'] + geom['crop_height']))
        x = torch.from_numpy(np.asarray(image, dtype=np.float32).copy()).permute(2, 0, 1) / 255.0
        h, w = x.shape[-2:]
        x = F.pad(x.unsqueeze(0), (0, (-w) % 14, 0, (-h) % 14), mode='replicate').squeeze(0)
        mean = x.new_tensor([.48145466, .4578275, .40821073])[:, None, None]
        std = x.new_tensor([.26862954, .26130258, .27577711])[:, None, None]
        return (x - mean) / std, int(row['label']), i


def save_csv(path, frame):
    path = Path(path)
    temp = path.with_suffix(path.suffix + '.tmp')
    frame.to_csv(temp, index=False)
    temp.replace(path)


def make_splits(df, cfg, seed, out):
    if cfg['data_mode'] == 'reuse_manifests':
        train, val, tests = reuse_saved_splits(df, cfg, seed, out)
    else:
        train, val, tests = _raw_splits(df, cfg, seed, out)
    columns = [c for c in ('sample_id', 'generator', 'label', 'pixel_hash') if c in val]
    plan = pd.concat([
        val[columns].reset_index(drop=True),
        pd.DataFrame([processing_parameters(sid, cfg, seed) for sid in val.sample_id]),
    ], axis=1)
    plan['batch_id'] = np.arange(len(plan)) // cfg['test_batch']
    plan['run_seed'] = seed
    plan['validation_processing_seed'] = cfg['validation_processing_seed']
    save_csv(out / 'validation_processing_plan.csv', plan)
    dump(out / 'validation_protocol.json', dict(
        cases=['native', 'processed'], monitor=VALIDATION_MONITOR,
        weights=dict(native=0.5, processed=0.5),
        processing_order=['decode_rgb', 'blur', 'jpeg', 'center_crop', 'edge_pad14', 'normalize'],
        validation_processing_seed=cfg['validation_processing_seed'], run_seed=seed,
        plan_hash=digest(plan.to_dict('records')),
        source='validation split from train folders only; official test excluded',
        batching='same fixed ID order and membership in both validation views',
        loss='mean per-image cross entropy computed directly from logits',
    ))
    print(f'[validation] Fixed native + processed views; plan: {out / "validation_processing_plan.csv"}', flush=True)
    return train, val, tests


@torch.no_grad()
def predict(model, anchor, frame, cfg, seed, device, case, return_loss=False):
    if len(frame) == 0:
        raise ValueError('Cannot evaluate an empty cohort')
    model.eval()
    result = np.full(len(frame), np.nan)
    batches = np.full(len(frame), -1, dtype=np.int64)
    loss_sum, count = 0.0, 0
    for bid, (images, labels, indices) in enumerate(loader(frame, cfg, seed, case=case)):
        # Both branches see exactly the SAME transformed tensors; no second RNG draw.
        bases = estimate_bases(anchor, images, cfg, device, model.selected) if anchor is not None else None
        for start, x in image_chunks(images, cfg['micro_batch']):
            with amp(cfg, device):
                logits = model(x.to(device), bases)
            if not torch.isfinite(logits).all():
                raise FloatingPointError(f'Nonfinite {case} logits')
            ix = indices[start:start + len(x)].numpy()
            result[ix] = logits.float().softmax(-1)[:, 1].cpu().numpy()
            batches[ix] = bid
            if return_loss:
                loss_sum += float(F.cross_entropy(
                    logits.float(), labels[start:start + len(x)].to(device), reduction='sum'))
                count += len(x)
        if bid % 25 == 0:
            print(f'[{case}] batch {bid + 1}/{math.ceil(len(frame) / cfg["test_batch"])}', flush=True)
    if not np.isfinite(result).all() or (batches < 0).any():
        raise ValueError('Missing/nonfinite prediction')
    if return_loss:
        if count != len(frame) or not math.isfinite(loss_sum):
            raise FloatingPointError('Incomplete/nonfinite validation loss')
        return result, batches, loss_sum / count
    return result, batches


def evaluate_validation(model, anchor, val, cfg, seed, device):
    summaries, predictions = {}, {}
    for case in ('native', 'processed'):
        p, batches, loss = predict(model, anchor, val, cfg, seed, device, case, return_loss=True)
        summaries[case] = dict(loss=loss, **metric(val.label.to_numpy(), p, cfg['threshold']))
        predictions[case] = (p, batches)
    if not np.array_equal(predictions['native'][1], predictions['processed'][1]):
        raise ValueError('Validation batch membership changed across views')
    monitor = (summaries['native']['loss'] + summaries['processed']['loss']) / 2.0
    return monitor, summaries, predictions


def train_model(model, anchor, train, val, cfg, seed, device, folder, signature):
    groups = [dict(params=model.vision.parameters(), lr=cfg['lr']),
              dict(params=model.head.parameters(), lr=cfg['head_lr'])]
    if len(model.gsd):
        groups.append(dict(params=model.gsd.parameters(), lr=cfg['modulation_lr']))
    optimizer = torch.optim.AdamW(groups, betas=tuple(cfg['betas']), eps=cfg['eps'], weight_decay=cfg['weight_decay'])
    scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda' and cfg['precision'] == 'fp16')
    stopper = EarlyStopping(cfg['early_stopping'], cfg['patience'], cfg['min_delta'])
    latest = folder / 'last.pt'
    start, best, history = 0, float('inf'), []
    if cfg['resume'] and latest.exists():
        ck = torch.load(latest, map_location='cpu', weights_only=False)
        if ck['signature'] != signature or ck.get('protocol') != protocol_id(cfg) or ck.get('config') != cfg:
            raise ValueError('Resume protocol/config/manifests changed; use a new output_root')
        model.load_state_dict(ck['model'])
        optimizer.load_state_dict(ck['optimizer'])
        scaler.load_state_dict(ck['scaler'])
        stopper.state.update(ck['early_stopping'])
        start, best, history = ck['epoch'] + 1, ck['best'], ck['history']
        del ck
        print(f'[resume] next epoch {start + 1}, stopped={stopper.state["stopped"]}', flush=True)
    for epoch in range(start, cfg['epochs']):
        if stopper.state['stopped']:
            break
        model.train()
        running, count, skipped = 0.0, 0, 0
        tick = time.monotonic()
        for bid, (images, labels, _) in enumerate(loader(train, cfg, seed, True, epoch)):
            bases = estimate_bases(anchor, images, cfg, device, model.selected) if anchor is not None else None
            optimizer.zero_grad(set_to_none=True)
            for a, x in image_chunks(images, cfg['micro_batch']):
                with amp(cfg, device):
                    logits = model(x.to(device), bases)
                    loss = F.cross_entropy(logits.float(), labels[a:a + len(x)].to(device), reduction='sum') / len(images)
                if not torch.isfinite(loss):
                    raise FloatingPointError('Nonfinite training loss')
                scaler.scale(loss).backward()
                running += float(loss.detach()) * len(images)
            old = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            skipped += int(scaler.get_scale() < old)
            count += len(images)
            if bid % 10 == 0:
                print(f'[epoch {epoch + 1}] batch {bid + 1}/{math.ceil(len(train) / cfg["train_batch"])}; loss={running / count:.5f}', flush=True)
        if count == 0:
            raise ValueError('Empty training cohort')
        vloss, summaries, predictions = evaluate_validation(model, anchor, val, cfg, seed, device)
        stop = stopper.update(vloss)
        entry = dict(epoch=epoch + 1, train_loss=running / count, val_loss=vloss,
                     val_native_loss=summaries['native']['loss'], val_processed_loss=summaries['processed']['loss'],
                     validation_monitor=VALIDATION_MONITOR,
                     val_native=summaries['native'], val_processed=summaries['processed'],
                     seconds=time.monotonic() - tick, amp_skipped_steps=skipped, train_images=count,
                     modulation={l: dict(a=float(g.a.detach()), b=float(g.b.detach())) for l, g in model.gsd.items()},
                     early_stopping=dict(stopper.state))
        history.append(entry)
        print(entry, flush=True)
        state = dict(model=model.state_dict(), epoch=epoch, signature=signature, config=cfg,
                     protocol=protocol_id(cfg), validation=summaries, val_loss=vloss)
        if vloss < best:
            best = vloss
            save_checkpoint(folder / 'best.pt', state)
        save_checkpoint(latest, dict(state, optimizer=optimizer.state_dict(), scaler=scaler.state_dict(),
                                     best=best, history=history, early_stopping=dict(stopper.state)))
        dump(folder / 'history.json', history)
        dump(folder / 'training_status.json', dict(epoch=epoch + 1, max_epochs=cfg['epochs'],
             validation_monitor=VALIDATION_MONITOR, val_loss=vloss, early_stopping=dict(stopper.state)))
        rows = [dict(epoch=e['epoch'], case=case, monitor_loss=e['val_loss'], **e['val_' + case])
                for e in history for case in ('native', 'processed')]
        save_csv(folder / 'validation_metrics.csv', pd.DataFrame(rows))
        columns = [c for c in ('sample_id', 'generator', 'label', 'pixel_hash') if c in val]
        frame = val[columns].copy()
        for case, (p, batches) in predictions.items():
            frame['prob_fake_' + case] = p
            frame['batch_id_' + case] = batches
        save_csv(folder / f'validation_predictions_epoch{epoch + 1:03d}.csv', frame)
        if stop:
            print('[early stopping] Restore best mean native/processed validation CE checkpoint.', flush=True)
            break
    ck = torch.load(folder / 'best.pt', map_location='cpu', weights_only=False)
    model.load_state_dict(ck['model'])
    dump(folder / 'selected_checkpoint.json', dict(epoch=ck['epoch'] + 1,
         criterion=VALIDATION_MONITOR, val_loss=ck['val_loss'], validation=ck['validation'], signature=signature))


def run(config=None):
    cfg = dict(DEFAULT_CONFIG, **(config or {}))
    validate_raw_config(cfg)
    print('[protocol] All7 crop560; train blur/JPEG resampled per epoch, SAME tensors for anchor/detector.', flush=True)
    print('[protocol] Fixed native/processed validation; early stopping = 0.5 * CE(native) + 0.5 * CE(processed).', flush=True)
    print('[protocol] New pretrained run only; do not reuse raw-only checkpoint/output directories.', flush=True)
    print(f'[protocol] Data mode: {cfg["data_mode"]}; manifests may be reused, detector weights are NOT reused.', flush=True)
    return _raw_run(cfg)


"""B-Free paired data layer for GSD training on ONE fake branch (torch-free).

Default branch: SD2.1_inpainted_samecat_origBG versus COCO_real_512 of the SAME COCO ID.
Everything here is deterministic and depends only on (seed, COCO ID, virtual epoch),
never on labels, worker order or global RNG state.
"""
import concurrent.futures
import hashlib
import io
import json
import math
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFilter

BFREE_REAL = 'COCO_real_512'
BFREE_ALL_BRANCHES = (
    'SD2.1_selfconditioned', 'SD2.1_selfconditioned_origBG',
    'SD2.1_inpainted_samecat', 'SD2.1_inpainted_samecat_origBG',
    'SD2.1_inpainted_diffcat', 'SD2.1_inpainted_diffcat_origBG',
)
BFREE_TRAIN_LIST_MD5 = '06581da53bf81741501396d71eb7ec5f'
BFREE_VALID_LIST_MD5 = '72e7b4be2d049029dbedca7e75af91b0'
BFREE_TRAIN_LIST_COUNT, BFREE_VALID_LIST_COUNT = 51517, 10304
CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)
# Keys that only affect where/how long a session runs; excluded from the run identity.
VOLATILE_KEYS = ('input_root', 'bfree_root', 'output_root', 'resume_from', 'time_budget_hours',
                 'num_workers', 'audit_workers', 'save_every_steps', 'log_every_steps',
                 'validation_time_reserve_minutes')


def sha(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str).encode()).hexdigest()


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(value, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
    temp.replace(path)


def atomic_csv(path, frame):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.tmp')
    frame.to_csv(temp, index=False)
    temp.replace(path)


def run_identity_hash(cfg):
    return sha({k: v for k, v in cfg.items() if k not in VOLATILE_KEYS})


# ----------------------------------------------------------------------------- dataset discovery
def candidate_mounts(input_root, max_depth=3):
    """Directories that may be a dataset root; never descend into image folders."""
    root = Path(input_root)
    if not root.is_dir():
        raise FileNotFoundError(f'Missing input_root {root}')
    found, pending = [], [(root, 0)]
    skip = {BFREE_REAL, *BFREE_ALL_BRANCHES}
    while pending:
        folder, depth = pending.pop()
        found.append(folder)
        if depth >= max_depth:
            continue
        try:
            children = sorted(p for p in folder.iterdir() if p.is_dir() and not p.is_symlink())
        except PermissionError:
            continue
        pending.extend((p, depth + 1) for p in children if p.name not in skip)
    return found


def find_bfree_root(cfg):
    if cfg.get('bfree_root'):
        candidates = [Path(cfg['bfree_root'])]
    else:
        candidates = candidate_mounts(cfg['input_root'], max_depth=4)
    matches = [p for p in candidates if (p / 'train_list.csv').is_file() and (p / 'valid_list.csv').is_file()
               and (p / BFREE_REAL).is_dir()]
    if len(matches) != 1:
        raise ValueError('Set bfree_root to the folder containing train_list.csv, valid_list.csv and '
                         f'{BFREE_REAL}. Found: {[str(p) for p in matches]}')
    return matches[0]


def read_bfree_list(path, expected_md5, expected_count, check_md5=True):
    raw = Path(path).read_bytes()
    checksum = hashlib.md5(raw).hexdigest()
    if check_md5 and checksum != expected_md5:
        raise ValueError(f'{path}: MD5 {checksum} differs from the official B-Free list')
    frame = pd.read_csv(io.BytesIO(raw), dtype=str, keep_default_na=False)
    if 'id' not in frame.columns:
        raise ValueError(f'{path}: missing id column')
    if expected_count is not None and len(frame) != expected_count:
        raise ValueError(f'{path}: expected {expected_count} rows, found {len(frame)}')
    if frame.id.duplicated().any() or not frame.id.str.fullmatch(r'\d{12}').all():
        raise ValueError(f'{path}: duplicated or invalid 12-digit COCO IDs')
    return frame, checksum


def bfree_folder(root, name, example_ids):
    """Single (root/name) or double-nested (root/name/name) layout, decided on several IDs."""
    root = Path(root)
    choices = []
    for folder in (root / name / name, root / name):
        hits = sum((folder / f'{i}.png').is_file() for i in example_ids)
        if hits:
            choices.append((hits, folder))
    if not choices:
        raise FileNotFoundError(f'Cannot find {name}/<id>.png (single or double nested) under {root}')
    choices.sort(key=lambda t: -t[0])
    if len(choices) > 1 and choices[0][0] == choices[1][0]:
        raise ValueError(f'Ambiguous layout for {name}: {[str(c[1]) for c in choices]}')
    return choices[0][1]


# ----------------------------------------------------------------------------- pair audit / mask
def load_rgb(path, side):
    with Image.open(path) as image:
        if image.format != 'PNG':
            raise ValueError(f'not PNG ({image.format})')
        if side is not None and image.size != (side, side):
            raise ValueError(f'size {image.size} != {side}x{side}')
        return np.asarray(image.convert('RGB'), dtype=np.uint8)


def pair_statistics(real_path, fake_path, side, diff_threshold):
    """Inpainted-region proxy from the paired difference. origBG pastes the real background back,
    so pixels outside the regenerated object should (almost) equal the real image."""
    try:
        real = load_rgb(real_path, side)
        fake = load_rgb(fake_path, side)
        if real.shape != fake.shape:
            raise ValueError(f'shape mismatch {real.shape} vs {fake.shape}')
        diff = np.abs(fake.astype(np.int16) - real.astype(np.int16)).max(axis=2)
        mask = diff > diff_threshold
        if mask.any():
            ys, xs = np.nonzero(mask)
            bbox = [int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1]
        else:
            bbox = [0, 0, 0, 0]
        mask_pixels, total_pixels = int(mask.sum()), int(mask.size)
        return dict(ok=True, error='', height=int(real.shape[0]), width=int(real.shape[1]),
                    changed_any_fraction=float((diff > 0).mean()),
                    mask_pixels=mask_pixels, total_pixels=total_pixels,
                    mask_fraction=mask_pixels / total_pixels, mean_abs_diff=float(diff.mean()),
                    bbox_x0=bbox[0], bbox_y0=bbox[1], bbox_x1=bbox[2], bbox_y1=bbox[3],
                    identical=bool(not diff.any()))
    except Exception as exc:  # recorded and counted, never silently used for training
        return dict(ok=False, error=f'{type(exc).__name__}: {exc}', height=-1, width=-1,
                    changed_any_fraction=float('nan'), mask_pixels=-1, total_pixels=-1,
                    mask_fraction=float('nan'),
                    mean_abs_diff=float('nan'), bbox_x0=-1, bbox_y0=-1, bbox_x1=-1, bbox_y1=-1,
                    identical=False)


def audit_pairs(ids, real_dir, fake_dir, cfg, cohort, log_every=5000):
    ids = list(ids)
    rows = [None] * len(ids)
    start = time.monotonic()

    def work(i):
        coco_id = ids[i]
        real_path, fake_path = Path(real_dir) / f'{coco_id}.png', Path(fake_dir) / f'{coco_id}.png'
        if not real_path.is_file() or not fake_path.is_file():
            stats = pair_statistics(real_path, fake_path, None, 0)  # produces the error row
            stats['error'] = 'missing ' + ('real' if not real_path.is_file() else 'fake')
            stats['ok'] = False
        else:
            stats = pair_statistics(real_path, fake_path, cfg['image_side'], cfg['mask_diff_threshold'])
        return i, dict(coco_id=coco_id, cohort=cohort, real_path=str(real_path), fake_path=str(fake_path), **stats)

    workers = max(1, int(cfg.get('audit_workers', 8)))
    done = 0
    with concurrent.futures.ThreadPoolExecutor(workers) as pool:
        for i, row in pool.map(work, range(len(ids)), chunksize=64):
            rows[i] = row
            done += 1
            if done % log_every == 0 or done == len(ids):
                rate = done / max(time.monotonic() - start, 1e-9)
                print(f'[audit:{cohort}] {done:,}/{len(ids):,} pairs; {rate:.0f} pairs/s', flush=True)
    return pd.DataFrame(rows)


def restore_audit_types(audit):
    """CSV round trip: rebuild booleans and the EXACT mask fraction from integer pixel counts."""
    audit = audit.copy()
    audit['coco_id'] = audit.coco_id.astype(str).str.zfill(12)
    for column in ('ok', 'eligible', 'identical'):
        if column in audit:
            audit[column] = audit[column].astype(str).str.lower() == 'true'
    pixels = pd.to_numeric(audit.mask_pixels, errors='coerce')
    total = pd.to_numeric(audit.total_pixels, errors='coerce')
    audit['mask_pixels'], audit['total_pixels'] = pixels, total
    audit['mask_fraction'] = np.where(audit.ok & (total > 0), pixels / total.where(total > 0, 1), np.nan)
    audit['exclusion_reason'] = audit.exclusion_reason.fillna('').astype(str)
    return audit


def mark_eligibility(audit, cfg):
    audit = audit.copy()
    reason = np.full(len(audit), '', dtype=object)
    reason[~audit.ok.to_numpy(bool)] = 'unreadable_or_missing'
    ok = audit.ok.to_numpy(bool)
    frac = audit.mask_fraction.to_numpy(float)
    reason[ok & (frac < cfg['min_mask_fraction'])] = 'mask_too_small_or_identical'
    reason[ok & (frac > cfg['max_mask_fraction'])] = 'mask_too_large_not_origBG_like'
    audit['exclusion_reason'] = reason
    audit['eligible'] = audit.exclusion_reason == ''
    return audit


def area_tertile_edges(fractions):
    q = np.quantile(np.asarray(fractions, float), [1 / 3, 2 / 3])
    return [float(q[0]), float(q[1])]


def area_group(fractions, edges):
    f = np.asarray(fractions, float)
    return np.where(f <= edges[0], 'small', np.where(f <= edges[1], 'medium', 'large'))


# ----------------------------------------------------------------------------- splits / batches
def id_rank(seed, purpose, coco_id):
    return int(sha([str(purpose), int(seed), str(coco_id)])[:16], 16)


def make_bfree_splits(train_audit, cfg, seed):
    """Validation stratified by mask-area tertile; the rest permuted ONCE into virtual epochs."""
    eligible = train_audit[train_audit.eligible].copy()
    if eligible.coco_id.duplicated().any():
        raise ValueError('Duplicate COCO IDs in the eligible train pool')
    edges = area_tertile_edges(eligible.mask_fraction)
    eligible['area_group'] = area_group(eligible.mask_fraction, edges)
    n_val = int(cfg['val_ids'])
    if n_val < 3 or n_val >= len(eligible):
        raise ValueError(f'val_ids={n_val} incompatible with {len(eligible)} eligible IDs')
    eligible['val_rank'] = [id_rank(seed, 'validation', i) for i in eligible.coco_id]
    quotas = {g: n_val // 3 + (1 if k < n_val % 3 else 0) for k, g in enumerate(('small', 'medium', 'large'))}
    val_parts = []
    for group, quota in quotas.items():
        members = eligible[eligible.area_group == group].sort_values('val_rank')
        if len(members) < quota:
            raise ValueError(f'Not enough {group} IDs for validation')
        val_parts.append(members.head(quota))
    val = pd.concat(val_parts)
    # Fixed evaluation order independent of the stratification.
    val = val.sort_values('val_rank').reset_index(drop=True)
    pool = eligible[~eligible.coco_id.isin(val.coco_id)].copy()
    pool['train_rank'] = [id_rank(seed, 'train_order', i) for i in pool.coco_id]
    pool = pool.sort_values('train_rank').reset_index(drop=True)
    ve_ids = int(cfg['ve_ids'])
    if ve_ids < 1 or ve_ids % cfg['pairs_per_batch']:
        raise ValueError('ve_ids must be a positive multiple of pairs_per_batch')
    n_ve = min(int(cfg['max_virtual_epochs']), len(pool) // ve_ids)
    if n_ve < 1:
        raise ValueError(f'Train pool {len(pool)} smaller than one virtual epoch ({ve_ids})')
    pool['virtual_epoch'] = -1
    pool.loc[:n_ve * ve_ids - 1, 'virtual_epoch'] = np.repeat(np.arange(n_ve), ve_ids)
    summary = dict(eligible=len(eligible), validation=len(val), train_pool=len(pool),
                   virtual_epochs=n_ve, ids_per_virtual_epoch=ve_ids,
                   images_per_virtual_epoch=2 * ve_ids, unused_reserve_ids=int((pool.virtual_epoch < 0).sum()),
                   area_tertile_edges=edges,
                   validation_area_groups=val.area_group.value_counts().to_dict(),
                   validation_order_hash=sha(val.coco_id.tolist()),
                   train_order_hash=sha(pool.coco_id.tolist()))
    if set(val.coco_id) & set(pool.coco_id):
        raise AssertionError('validation/train overlap')
    return val, pool, summary


def pair_rows(frame):
    """Interleave real/fake per ID: row 2k is real (label0), row 2k+1 is fake (label1)."""
    rows = []
    for r in frame.itertuples(index=False):
        rows.append(dict(coco_id=r.coco_id, label=0, path=r.real_path))
        rows.append(dict(coco_id=r.coco_id, label=1, path=r.fake_path))
    return rows


def pair_batches(n_ids, pairs_per_batch):
    """Logical batches of row indices: each batch = `pairs_per_batch` complete real/fake pairs."""
    if n_ids % pairs_per_batch:
        raise ValueError('Number of IDs must be a multiple of pairs_per_batch (complete pairs only)')
    return [list(range(2 * s, 2 * (s + pairs_per_batch))) for s in range(0, n_ids, pairs_per_batch)]


# ----------------------------------------------------------------------------- processing
def pair_processing_parameters(coco_id, cfg, seed, train=False, virtual_epoch=0):
    """Same blur/JPEG draw for the real AND fake image of one COCO ID.
    Train: redrawn every virtual epoch. Validation: fixed by validation_processing_seed."""
    identity = ['bfree_pair_blur_jpeg_v1', int(seed), str(coco_id)]
    identity += ['train', int(virtual_epoch)] if train else ['validation', int(cfg['validation_processing_seed'])]
    rng = random.Random(int(sha(identity)[:16], 16))
    blur_applied = rng.random() < cfg['blur_probability']
    jpeg_applied = rng.random() < cfg['jpeg_probability']
    sigma = rng.uniform(*cfg['blur_sigma'])
    quality = rng.randint(*cfg['jpeg_train_quality'])
    return dict(blur_applied=blur_applied, blur_sigma=sigma if blur_applied else None,
                jpeg_applied=jpeg_applied, jpeg_quality=quality if jpeg_applied else None)


def jpeg_roundtrip(image, quality, subsampling):
    # Byte-for-byte the same call as runtime.jpeg used by GSD-aug-Tiny.
    buf = io.BytesIO()
    image.save(buf, format='JPEG', quality=int(quality), subsampling=subsampling,
               optimize=False, progressive=False)
    buf.seek(0)
    with Image.open(buf) as decoded:
        return decoded.convert('RGB')


def apply_pair_processing(image, parameters, cfg):
    # Same order as GSD-aug-Tiny: decoded RGB -> blur -> JPEG, BEFORE crop/pad/normalize.
    if parameters['blur_applied']:
        image = image.filter(ImageFilter.GaussianBlur(parameters['blur_sigma']))
    if parameters['jpeg_applied']:
        image = jpeg_roundtrip(image, parameters['jpeg_quality'], cfg['jpeg_subsampling'])
    return image


def crop_geometry(width, height, crop_max_side):
    # Identical to runtime_raw_all7.input_geometry for center_crop_max_edge_pad14.
    cw, ch = min(width, crop_max_side), min(height, crop_max_side)
    left, top = (width - cw) // 2, (height - ch) // 2
    pw, ph = ((cw + 13) // 14) * 14, ((ch + 13) // 14) * 14
    return dict(crop_left=left, crop_top=top, crop_width=cw, crop_height=ch,
                input_width=pw, input_height=ph, tokens=(pw // 14) * (ph // 14) + 1)


def load_processed_crop(path, cfg, parameters=None):
    """Decoded RGB (+ optional pair processing) cropped to <=crop_max_side; pad/normalize happen in torch."""
    with Image.open(path) as source:
        if source.format != 'PNG':
            raise ValueError(f'{path}: expected PNG')
        image = source.convert('RGB')
    if parameters is not None:
        image = apply_pair_processing(image, parameters, cfg)
    g = crop_geometry(*image.size, cfg['crop_max_side'])
    return image.crop((g['crop_left'], g['crop_top'], g['crop_left'] + g['crop_width'], g['crop_top'] + g['crop_height']))


# ----------------------------------------------------------------------------- resume discovery
def find_previous_outputs(cfg, run_hash, exclude=()):
    """Previous sessions' outputs attached as Input: folders with run_state.json of the same run."""
    exclude = {os.path.normcase(os.path.abspath(p)) for p in exclude}
    matches = []
    for folder in candidate_mounts(cfg['input_root'], max_depth=5):
        if os.path.normcase(os.path.abspath(folder)) in exclude:
            continue
        state_path = folder / 'run_state.json'
        if state_path.is_file():
            try:
                state = json.loads(state_path.read_text(encoding='utf-8'))
            except Exception:
                continue
            if state.get('run_hash') == run_hash:
                matches.append((int(state.get('images_seen', 0)), str(state.get('updated', '')), folder, state))
    matches.sort(key=lambda t: (t[0], t[1]))
    return matches


"""GSD training on ONE B-Free branch (default SD2.1_inpainted_samecat_origBG), paired with COCO real.

Appended after runtime.py, runtime_raw_all7.py, manifest_reuse.py, runtime_aug_all7.py and
bfree_data.py. Model, GSD basis, optimizer, precision and input geometry are the SAME code and
values as GSD-aug-Tiny (train_gsd_all7_crop560_blur_jpeg_kaggle.ipynb). What changes:
  * data: B-Free train_list minus valid_list, one fake branch, real/fake of the same COCO ID;
  * batch: 8 = 4 COCO IDs x (real, fake) so the GSD basis is estimated on content-matched pairs;
  * blur/JPEG: SAME recipe, but one shared draw per pair (seed, COCO ID, virtual epoch);
  * schedule: virtual epochs of 12,600 IDs (25,200 images = one Tiny epoch), no ID repeated,
    validation + early stopping after each, mid-epoch resume, Kaggle time budget.
Checkpoints use the evaluator payload format so the existing Tiny 9-case and B-Free-valid
evaluators can score them with identical inference code.
"""
import shutil
import time

BFREE_SESSION_STARTED = time.time()
BFREE_TRAIN_PROTOCOL = 'gsd_bfree_single_branch_paired_v1'
INFERENCE_KEYS = ('model_id', 'input_mode', 'crop_max_side', 'test_batch', 'micro_batch', 'anchor_micro_batch',
                  'precision', 'basis_method', 'attention', 'pooling', 'k', 'lambda_max', 'a_init', 'b_init',
                  'jpeg_subsampling', 'threshold')

BFREE_TRAIN_DEFAULTS = dict(
    training_protocol=BFREE_TRAIN_PROTOCOL,
    input_root='/kaggle/input', bfree_root=None,
    output_root='/kaggle/working/gsd_bfree_samecat_origbg_v1', resume_from='auto',
    train_branch='SD2.1_inpainted_samecat_origBG', real_folder=BFREE_REAL,
    check_list_md5=True, image_side=512,
    mask_diff_threshold=8, min_mask_fraction=0.001, max_mask_fraction=0.95,
    max_unreadable_fraction=0.01, audit_valid_ids=True, audit_workers=8,
    seed=42, val_ids=2000, ve_ids=12600, max_virtual_epochs=3, pairs_per_batch=4,
    # ---- identical to GSD-aug-Tiny -------------------------------------------------------
    model_id='openai/clip-vit-large-patch14', local_files_only=False,
    input_mode='center_crop_max_edge_pad14', crop_max_side=560,
    train_batch=8, test_batch=8, micro_batch=1, anchor_micro_batch=1,
    lr=1e-6, head_lr=1e-6, modulation_lr=1e-6, weight_decay=0.01, betas=[0.9, 0.999], eps=1e-8,
    k=64, lambda_max=1.0, a_init=2.0, b_init=1.0, precision='fp16', gradient_checkpointing=True,
    attention='sdpa', basis_method='gram_eigh', pooling='patch_mean_post_ln', threshold=0.5,
    jpeg_probability=0.5, jpeg_train_quality=[75, 100], jpeg_subsampling=0,
    blur_probability=0.5, blur_sigma=[0.0, 2.0], horizontal_flip_probability=0.0,
    validation_processing_seed=42, validation_monitor=VALIDATION_MONITOR,
    early_stopping=True, patience=2, min_delta=0.001,
    # ---- session handling ------------------------------------------------------------------
    num_workers=2, save_every_steps=250, log_every_steps=50, time_budget_hours=11.0,
    validation_time_reserve_minutes=None, save_virtual_epoch_checkpoints=True,
    keep_last_on_complete=False, smoke=False, allow_cpu=False,
)


def bfree_smoke_overrides(cfg):
    return dict(cfg, smoke=True, output_root=cfg['output_root'].rstrip('/') + '_smoke',
                val_ids=8, ve_ids=8, max_virtual_epochs=2, audit_valid_ids=False,
                save_every_steps=1, log_every_steps=1)


def validate_bfree_config(cfg):
    unknown = set(cfg) - set(BFREE_TRAIN_DEFAULTS)
    if unknown:
        raise ValueError(f'Unknown config keys: {sorted(unknown)}')
    if cfg['training_protocol'] != BFREE_TRAIN_PROTOCOL:
        raise ValueError('training_protocol is fixed')
    if cfg['train_branch'] not in BFREE_ALL_BRANCHES:
        raise ValueError(f'train_branch must be one of {BFREE_ALL_BRANCHES}')
    if cfg['train_batch'] != 2 * cfg['pairs_per_batch'] or cfg['test_batch'] != cfg['train_batch']:
        raise ValueError('Logical batch must be 2 x pairs_per_batch (complete real/fake pairs) for train and eval')
    if cfg['val_ids'] % cfg['pairs_per_batch'] or cfg['ve_ids'] % cfg['pairs_per_batch']:
        raise ValueError('val_ids and ve_ids must be multiples of pairs_per_batch')
    if cfg['horizontal_flip_probability'] != 0:
        raise ValueError('P2 keeps the GSD-aug-Tiny recipe: no flip')
    if not 0 < cfg['time_budget_hours'] <= 11.5:
        raise ValueError('time_budget_hours must be in (0, 11.5] for a 12 h Kaggle session')
    # Re-use the GSD-aug-Tiny augmentation/geometry/batch validator on an equivalent config.
    validate_raw_config(dict(DEFAULT_CONFIG, **{k: cfg[k] for k in (
        'input_mode', 'crop_max_side', 'train_batch', 'test_batch', 'micro_batch', 'anchor_micro_batch',
        'k', 'precision', 'attention', 'basis_method', 'pooling', 'jpeg_probability', 'jpeg_train_quality',
        'jpeg_subsampling', 'blur_probability', 'blur_sigma', 'horizontal_flip_probability',
        'validation_processing_seed', 'validation_monitor')}, seeds=[cfg['seed']], epochs=1))


def eval_compat_config(cfg):
    """Config the EXISTING evaluators read. Inference keys are copied from this run; the
    train_generators/expected_generators fields only describe the Tiny TEST cohort layout."""
    compat = dict(DEFAULT_CONFIG)
    compat.update({k: cfg[k] for k in INFERENCE_KEYS})
    compat.update(local_files_only=cfg['local_files_only'], gradient_checkpointing=cfg['gradient_checkpointing'],
                  train_batch=cfg['train_batch'], seeds=[cfg['seed']], models=['clip_gsd'], smoke=bool(cfg['smoke']),
                  eval_cases=['native'], manifest_root=None, dataset_root=None, data_mode='reuse_manifests',
                  output_root=cfg['output_root'], lr=cfg['lr'], head_lr=cfg['head_lr'],
                  modulation_lr=cfg['modulation_lr'], weight_decay=cfg['weight_decay'],
                  jpeg_probability=cfg['jpeg_probability'], blur_probability=cfg['blur_probability'],
                  jpeg_train_quality=cfg['jpeg_train_quality'], blur_sigma=cfg['blur_sigma'],
                  horizontal_flip_probability=0.0,
                  compat_note=('Evaluator view of a B-Free-trained checkpoint. Training data: '
                               f"B-Free {cfg['train_branch']} vs {cfg['real_folder']}. "
                               'train_generators here only describes the Tiny TEST manifest.'))
    return compat


def check_evaluator_compatibility(payload, cfg):
    """Mirror of evaluate_uploaded_best.validate_uploaded_payload, applied to the compat view."""
    if not isinstance(payload, dict) or not {'model', 'config', 'epoch', 'signature'} <= set(payload):
        raise ValueError('Checkpoint must contain model/config/epoch/signature')
    compat = payload['config']['eval_compat_config']
    _raw_validate(dict(compat, jpeg_probability=0., blur_probability=0., horizontal_flip_probability=0.))
    if compat['input_mode'] != 'center_crop_max_edge_pad14' or compat.get('crop_max_side') != 560:
        raise ValueError('Evaluators expect the crop560 protocol')
    if len(compat['seeds']) != 1 or compat['test_batch'] != 8:
        raise ValueError('Evaluators expect one seed and logical test batch 8')
    for key in INFERENCE_KEYS:
        if compat[key] != payload['config'][key]:
            raise ValueError(f'compat/training mismatch for {key}')
    if not isinstance(payload['model'], dict) or not payload['model']:
        raise ValueError('Missing model state_dict')
    return True


# ----------------------------------------------------------------------------- data (torch)
class BFreePairImages(Dataset):
    """Rows interleave real/fake of the same ID. mode: 'train' | 'native' | 'processed'."""
    def __init__(self, rows, cfg, seed, mode, virtual_epoch=0):
        if mode not in ('train', 'native', 'processed'):
            raise ValueError(mode)
        self.rows, self.cfg, self.seed, self.mode, self.ve = rows, cfg, seed, mode, virtual_epoch

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        row = self.rows[i]
        if self.mode == 'native':
            parameters = None
        else:
            parameters = pair_processing_parameters(row['coco_id'], self.cfg, self.seed,
                                                    train=self.mode == 'train', virtual_epoch=self.ve)
        image = load_processed_crop(row['path'], self.cfg, parameters)
        # Tensor path identical to GSD-aug-Tiny Images.__getitem__.
        x = torch.from_numpy(np.asarray(image, dtype=np.float32).copy()).permute(2, 0, 1) / 255.0
        h, w = x.shape[-2:]
        x = F.pad(x.unsqueeze(0), (0, (-w) % 14, 0, (-h) % 14), mode='replicate').squeeze(0)
        mean = x.new_tensor([.48145466, .4578275, .40821073])[:, None, None]
        std = x.new_tensor([.26862954, .26130258, .27577711])[:, None, None]
        return (x - mean) / std, int(row['label']), i


def pair_loader(rows, batches, cfg, seed, mode, virtual_epoch=0):
    return DataLoader(BFreePairImages(rows, cfg, seed, mode, virtual_epoch), batch_sampler=batches,
                      num_workers=cfg['num_workers'], collate_fn=native_collate,
                      pin_memory=torch.cuda.is_available())


def check_pair_batch(rows, indices):
    ids = [rows[i]['coco_id'] for i in indices]
    labels = [rows[i]['label'] for i in indices]
    if labels != [0, 1] * (len(indices) // 2) or ids[0::2] != ids[1::2]:
        raise AssertionError('Logical batch is not made of complete real/fake pairs of the same ID')


@torch.no_grad()
def evaluate_pairs(model, anchor, frame, cfg, seed, device, mode):
    model.eval()
    rows = pair_rows(frame)
    batches = pair_batches(len(frame), cfg['pairs_per_batch'])
    probs = np.full(len(rows), np.nan)
    margins = np.full(len(rows), np.nan)
    loss_sum, started = 0.0, time.monotonic()
    for bid, (images, labels, indices) in enumerate(pair_loader(rows, batches, cfg, seed, mode)):
        check_pair_batch(rows, indices.tolist())
        bases = estimate_bases(anchor, images, cfg, device, model.selected)
        for start, x in image_chunks(images, cfg['micro_batch']):
            with amp(cfg, device):
                logits = model(x.to(device), bases)
            logits = logits.float()
            if not torch.isfinite(logits).all():
                raise FloatingPointError(f'Nonfinite {mode} validation logits')
            ix = indices[start:start + len(x)].numpy()
            probs[ix] = logits.softmax(-1)[:, 1].cpu().numpy()
            margins[ix] = (logits[:, 1] - logits[:, 0]).cpu().numpy()
            loss_sum += float(F.cross_entropy(logits, labels[start:start + len(x)].to(device), reduction='sum'))
        if bid % 100 == 0:
            print(f'[val:{mode}] batch {bid + 1}/{len(batches)}', flush=True)
    if not np.isfinite(probs).all():
        raise ValueError('Missing validation predictions')
    seconds = time.monotonic() - started
    return probs, margins, loss_sum / len(rows), seconds / len(rows)


def area_metrics(frame, probs, threshold):
    """Fake recall / AUC by mask-area group (reals shared). Rows: real at 2k, fake at 2k+1."""
    real_p, fake_p = probs[0::2], probs[1::2]
    out = {}
    for group in ('small', 'medium', 'large'):
        sel = (frame.area_group == group).to_numpy()
        if sel.sum() == 0:
            continue
        y = np.r_[np.zeros(len(real_p)), np.ones(sel.sum())]
        p = np.r_[real_p, fake_p[sel]]
        out[group] = dict(n_fake=int(sel.sum()), fake_recall=float((fake_p[sel] >= threshold).mean()),
                          auc_vs_all_real=float(roc_auc_score(y, p)))
    return out


# ----------------------------------------------------------------------------- checkpoints
def evaluator_payload(model, cfg, signature, ve, summaries, val_loss, images_seen):
    config = dict(cfg)
    config['eval_compat_config'] = eval_compat_config(cfg)
    return dict(model={k: v.detach().to('cpu') for k, v in model.state_dict().items()},
                epoch=int(ve), virtual_epoch=int(ve) + 1, signature=signature, config=config,
                protocol=BFREE_TRAIN_PROTOCOL, validation=summaries, val_loss=float(val_loss),
                images_seen=int(images_seen))


def save_and_verify_payload(path, payload, cfg):
    save_checkpoint(Path(path), payload)
    loaded = torch.load(path, map_location='cpu', weights_only=True, mmap=True)
    check_evaluator_compatibility(loaded, cfg)
    if loaded['signature'] != payload['signature'] or set(loaded['model']) != set(payload['model']):
        raise ValueError(f'Round-trip check failed for {path}')
    del loaded
    print(f'[checkpoint] {Path(path).name} saved; weights_only reload + evaluator checks OK', flush=True)


def write_state(out, state):
    state = dict(state, updated=time.strftime('%Y-%m-%dT%H:%M:%S'))
    atomic_json(out / 'run_state.json', state)
    return state


def save_last(out, model, optimizer, scaler, stopper, state):
    started = time.monotonic()
    save_checkpoint(out / 'last.pt', dict(model=model.state_dict(), optimizer=optimizer.state_dict(),
                                          scaler=scaler.state_dict(), early_stopping=dict(stopper.state),
                                          run_state=dict(state)))
    state = write_state(out, state)
    print(f"[save] last.pt at VE{state['ve'] + 1} step {state['step']} ({state['phase']}); "
          f"{time.monotonic() - started:.0f}s", flush=True)
    return state


# ----------------------------------------------------------------------------- preparation
def copy_previous_output(cfg, run_hash, out):
    if cfg['resume_from'] is None:
        return None
    if cfg['resume_from'] == 'auto':
        matches = find_previous_outputs(cfg, run_hash, exclude=[out])
    else:
        folder = Path(cfg['resume_from'])
        state = json.loads((folder / 'run_state.json').read_text(encoding='utf-8'))
        if state.get('run_hash') != run_hash:
            raise ValueError(f'{folder}: run_state belongs to a different config')
        matches = [(int(state.get('images_seen', 0)), '', folder, state)]
    if not matches:
        print('[resume] No previous output of this run attached; starting fresh.', flush=True)
        return None
    images, _, folder, state = matches[-1]
    local = out / 'run_state.json'
    if local.is_file():
        mine = json.loads(local.read_text(encoding='utf-8'))
        if int(mine.get('images_seen', 0)) >= images:
            print('[resume] Local output is already at least as advanced; not copying.', flush=True)
            return None
    print(f'[resume] Copying previous output {folder} (images_seen={images:,}, status={state.get("status")})', flush=True)
    shutil.copytree(folder, out, dirs_exist_ok=True)
    return str(folder)


def prepare_bfree_data(cfg, out):
    root = find_bfree_root(cfg)
    train_list, train_md5 = read_bfree_list(root / 'train_list.csv', BFREE_TRAIN_LIST_MD5,
                                            BFREE_TRAIN_LIST_COUNT if cfg['check_list_md5'] else None,
                                            cfg['check_list_md5'])
    valid_list, valid_md5 = read_bfree_list(root / 'valid_list.csv', BFREE_VALID_LIST_MD5,
                                            BFREE_VALID_LIST_COUNT if cfg['check_list_md5'] else None,
                                            cfg['check_list_md5'])
    valid_ids = set(valid_list.id)
    if not valid_ids <= set(train_list.id):
        raise ValueError('valid_list must be contained in train_list (official release)')
    train_ids = [i for i in train_list.id if i not in valid_ids]  # 41,213 for the official lists
    real_dir = bfree_folder(root, cfg['real_folder'], train_ids[:20])
    fake_dir = bfree_folder(root, cfg['train_branch'], train_ids[:20])
    meta = dict(bfree_root=str(root), real_dir=str(real_dir), fake_dir=str(fake_dir),
                train_list_md5=train_md5, valid_list_md5=valid_md5,
                train_list_ids=len(train_list), valid_list_ids=len(valid_list), train_pool_ids=len(train_ids),
                branch=cfg['train_branch'], audit_rule={k: cfg[k] for k in (
                    'image_side', 'mask_diff_threshold', 'min_mask_fraction', 'max_mask_fraction')})
    if cfg['smoke']:
        train_ids = sorted(train_ids, key=lambda i: id_rank(cfg['seed'], 'smoke', i))[:64]
        meta['smoke_subset'] = len(train_ids)
    audit_path, meta_path = out / 'audit_train.csv', out / 'audit_meta.json'
    reuse = False
    if audit_path.is_file() and meta_path.is_file():
        old = json.loads(meta_path.read_text(encoding='utf-8'))
        same = {k: old.get(k) for k in ('train_list_md5', 'valid_list_md5', 'branch', 'audit_rule', 'smoke_subset')} == \
               {k: meta.get(k) for k in ('train_list_md5', 'valid_list_md5', 'branch', 'audit_rule', 'smoke_subset')}
        reuse = same
    if reuse:
        audit = restore_audit_types(pd.read_csv(audit_path, dtype={'coco_id': str, 'exclusion_reason': str},
                                                keep_default_na=False))
        audit['real_path'] = [str(real_dir / f'{i}.png') for i in audit.coco_id]
        audit['fake_path'] = [str(fake_dir / f'{i}.png') for i in audit.coco_id]
        if sorted(audit.coco_id) != sorted(train_ids):
            raise ValueError('Saved audit IDs differ from the current train pool')
        print(f'[audit] Reusing saved audit of {len(audit):,} train pairs (paths rebased).', flush=True)
    else:
        print(f'[audit] Decoding {len(train_ids):,} train pairs ({cfg["real_folder"]} vs {cfg["train_branch"]}); '
              'mask = max-channel |fake-real| > threshold.', flush=True)
        audit = mark_eligibility(audit_pairs(train_ids, real_dir, fake_dir, cfg, 'train'), cfg)
        atomic_csv(audit_path, audit)
        atomic_json(meta_path, meta)
    unreadable = float((~audit.ok).mean())
    if unreadable > cfg['max_unreadable_fraction']:
        raise ValueError(f'{unreadable:.2%} of train pairs missing/unreadable: wrong dataset version/layout? '
                         f'See {audit_path}')
    ok = audit[audit.ok]
    summary = dict(pairs=len(audit), unreadable=int((~audit.ok).sum()), eligible=int(audit.eligible.sum()),
                   exclusions=audit.exclusion_reason[audit.exclusion_reason != ''].value_counts().to_dict(),
                   identical_pairs=int(ok.identical.astype(bool).sum()) if len(ok) else 0,
                   mask_fraction_quantiles={str(q): float(ok.mask_fraction.quantile(q)) for q in
                                            (0, .01, .05, .1, .25, .5, .75, .9, .95, .99, 1)} if len(ok) else {},
                   median_changed_any_minus_mask=float((pd.to_numeric(ok.changed_any_fraction) -
                                                        ok.mask_fraction).median()) if len(ok) else None)
    atomic_json(out / 'audit_summary.json', summary)
    print('[audit] ' + json.dumps(summary, indent=1), flush=True)
    valid_path = out / 'audit_valid.csv'
    if cfg['audit_valid_ids'] and not valid_path.is_file():
        vfake = bfree_folder(root, cfg['train_branch'], valid_list.id.tolist()[:20])
        print(f'[audit] Mask areas for {len(valid_list):,} official valid pairs (analysis only, never trained on).',
              flush=True)
        atomic_csv(valid_path, audit_pairs(valid_list.id.tolist(), real_dir, vfake, cfg, 'valid'))
    return audit, meta


# ----------------------------------------------------------------------------- main
def run_bfree_training(config=None, build=None):
    cfg = dict(BFREE_TRAIN_DEFAULTS, **(config or {}))
    if cfg['smoke'] is True and not cfg['output_root'].endswith('_smoke'):
        cfg = bfree_smoke_overrides(cfg)
    validate_bfree_config(cfg)
    deadline = BFREE_SESSION_STARTED + cfg['time_budget_hours'] * 3600
    out = Path(cfg['output_root'])
    out.mkdir(parents=True, exist_ok=True)
    run_hash = run_identity_hash(cfg)
    restored = copy_previous_output(cfg, run_hash, out)
    if (out / 'config.json').is_file():
        previous = json.loads((out / 'config.json').read_text(encoding='utf-8'))
        if run_identity_hash(previous) != run_hash:
            raise ValueError('output_root holds a different run; choose a new output_root')
    if (out / 'run_state.json').is_file():
        saved = json.loads((out / 'run_state.json').read_text(encoding='utf-8'))
        if saved.get('run_hash') == run_hash and saved.get('status') == 'complete':
            print('[done] This run is already complete (see completion.json); nothing to train.', flush=True)
            return finish_report(out, saved)
    atomic_json(out / 'config.json', cfg)
    seed = cfg['seed']
    print(f"[protocol] GSD on B-Free {cfg['train_branch']} vs {cfg['real_folder']}; batch8 = 4 COCO-ID pairs; "
          'blur/JPEG shared per pair; crop560/pad14; recipe = GSD-aug-Tiny.', flush=True)

    audit, meta = prepare_bfree_data(cfg, out)
    val, pool, split_summary = make_bfree_splits(audit, cfg, seed)
    atomic_csv(out / 'validation_ids.csv', val[['coco_id', 'mask_fraction', 'area_group']])
    atomic_csv(out / 'train_order.csv', pool[['coco_id', 'virtual_epoch', 'mask_fraction', 'area_group']])
    atomic_json(out / 'split_summary.json', split_summary)
    print('[split] ' + json.dumps(split_summary), flush=True)
    signature = sha(dict(run_hash=run_hash, validation=split_summary['validation_order_hash'],
                         train=split_summary['train_order_hash'], lists=[meta['train_list_md5'], meta['valid_list_md5']],
                         eligible=sha(audit[audit.eligible][['coco_id']].assign(
                             m=audit[audit.eligible].mask_fraction.round(6)).values.tolist())))
    n_ve = split_summary['virtual_epochs']

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if device.type != 'cuda' and not cfg['allow_cpu']:
        raise RuntimeError('Select a Kaggle GPU (T4 / P100). This notebook does not run on CPU/TPU.')
    if device.type == 'cuda':
        if cfg['precision'] == 'bf16' and not torch.cuda.is_bf16_supported():
            raise ValueError('GPU does not support bf16; keep fp16')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
    seed_all(seed)
    model, anchor = (build or build_models)(cfg, device, True)
    groups = [dict(params=model.vision.parameters(), lr=cfg['lr']),
              dict(params=model.head.parameters(), lr=cfg['head_lr']),
              dict(params=model.gsd.parameters(), lr=cfg['modulation_lr'])]
    optimizer = torch.optim.AdamW(groups, betas=tuple(cfg['betas']), eps=cfg['eps'], weight_decay=cfg['weight_decay'])
    scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda' and cfg['precision'] == 'fp16')
    stopper = EarlyStopping(cfg['early_stopping'], cfg['patience'], cfg['min_delta'])
    state = dict(run_hash=run_hash, signature=signature, status='running', ve=0, phase='train', step=0,
                 images_seen=0, best_val_loss=None, best_ve=None, virtual_epochs=n_ve, history=[],
                 train_seconds_per_image=None, eval_seconds_per_image=None, sessions=[])
    if (out / 'last.pt').is_file():
        ck = torch.load(out / 'last.pt', map_location='cpu', weights_only=False)
        if ck['run_state']['signature'] != signature:
            raise ValueError('last.pt signature differs from this data/config; use a new output_root')
        model.load_state_dict(ck['model'])
        optimizer.load_state_dict(ck['optimizer'])
        scaler.load_state_dict(ck['scaler'])
        stopper.state.update(ck['early_stopping'])
        state = dict(ck['run_state'])
        if state['status'] != 'complete':
            state['status'] = 'running'
        del ck
        print(f"[resume] VE{state['ve'] + 1} step {state['step']} phase={state['phase']}; "
              f"images_seen={state['images_seen']:,}", flush=True)
    state['sessions'] = list(state.get('sessions', [])) + [dict(started=time.strftime('%Y-%m-%dT%H:%M:%S'),
                                                                 restored_from=restored)]
    state = write_state(out, state)
    if state['status'] == 'complete':
        print('[done] This run is already complete; nothing to train.', flush=True)
        return finish_report(out, state)

    save_reserve = 10 * 60
    while state['ve'] < n_ve and state['status'] != 'complete':
        ve = state['ve']
        frame = pool[pool.virtual_epoch == ve].reset_index(drop=True)
        if state['phase'] == 'train':
            rows = pair_rows(frame)
            batches = pair_batches(len(frame), cfg['pairs_per_batch'])
            start_step = state['step']
            print(f'[train] VE{ve + 1}/{n_ve}: {len(frame):,} IDs = {len(rows):,} images, '
                  f'{len(batches):,} steps, starting at step {start_step}', flush=True)
            model.train()
            tick, window_loss, window_n, skipped = time.monotonic(), 0.0, 0, 0
            log_rows = []
            loader = pair_loader(rows, batches[start_step:], cfg, seed, 'train', ve)
            for offset, (images, labels, indices) in enumerate(loader):
                step = start_step + offset
                if time.time() > deadline - save_reserve:
                    state.update(step=step, status='paused_time_budget')
                    append_train_log(out, log_rows)
                    state = save_last(out, model, optimizer, scaler, stopper, state)
                    print('[session] Time budget reached mid virtual epoch. Save Version, then attach this '
                          'Output as Input and run again: training resumes exactly here.', flush=True)
                    return finish_report(out, state)
                check_pair_batch(rows, indices.tolist())
                step_started = time.monotonic()
                bases = estimate_bases(anchor, images, cfg, device, model.selected)
                optimizer.zero_grad(set_to_none=True)
                batch_loss = 0.0
                for a, x in image_chunks(images, cfg['micro_batch']):
                    with amp(cfg, device):
                        logits = model(x.to(device), bases)
                        loss = F.cross_entropy(logits.float(), labels[a:a + len(x)].to(device),
                                               reduction='sum') / len(images)
                    if not torch.isfinite(loss):
                        raise FloatingPointError('Nonfinite training loss')
                    scaler.scale(loss).backward()
                    batch_loss += float(loss.detach()) * len(images)
                old_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                skipped += int(scaler.get_scale() < old_scale)
                state['images_seen'] += len(images)
                window_loss += batch_loss
                window_n += len(images)
                spi = (time.monotonic() - step_started) / len(images)
                prev = state.get('train_seconds_per_image')
                state['train_seconds_per_image'] = spi if prev is None else 0.98 * prev + 0.02 * spi
                done = step + 1
                if done % cfg['log_every_steps'] == 0 or done == len(batches):
                    eta_h = (len(batches) - done) * len(images) * state['train_seconds_per_image'] / 3600
                    log_rows.append(dict(ve=ve + 1, step=done, images_seen=state['images_seen'],
                                         loss=window_loss / window_n, sec_per_image=state['train_seconds_per_image'],
                                         amp_skipped=skipped))
                    print(f'[VE{ve + 1}] step {done}/{len(batches)} loss={window_loss / window_n:.5f} '
                          f'{state["train_seconds_per_image"]:.3f}s/img ETA {eta_h:.2f}h', flush=True)
                    window_loss, window_n = 0.0, 0
                if done % cfg['save_every_steps'] == 0 and done < len(batches):
                    state['step'] = done
                    append_train_log(out, log_rows)
                    log_rows = []
                    state = save_last(out, model, optimizer, scaler, stopper, state)
            append_train_log(out, log_rows)
            state.update(phase='validate', step=0)
            state['ve_train_seconds'] = time.monotonic() - tick
            state = save_last(out, model, optimizer, scaler, stopper, state)

        # ---------------- validation for this virtual epoch
        per_image = state.get('eval_seconds_per_image') or (
            0.45 * state['train_seconds_per_image'] if state.get('train_seconds_per_image') else 0.3)
        needed = 2 * 2 * len(val) * per_image * 1.15 + save_reserve
        if cfg['validation_time_reserve_minutes'] is not None:
            needed = cfg['validation_time_reserve_minutes'] * 60
        if time.time() + needed > deadline:
            state['status'] = 'paused_time_budget'
            state = write_state(out, state)
            print(f'[session] Not enough time for validation (~{needed / 60:.0f} min). Save Version, attach '
                  'this Output as Input and run again: validation of this VE runs first.', flush=True)
            return finish_report(out, state)
        summaries, frames = {}, val.copy()
        vtick = time.monotonic()
        for mode in ('native', 'processed'):
            p, m, loss, spi = evaluate_pairs(model, anchor, val, cfg, seed, device, mode)
            y = np.tile([0, 1], len(val))
            summaries[mode] = dict(loss=float(loss), **metric(y, p, cfg['threshold']),
                                   by_area=area_metrics(val, p, cfg['threshold']))
            frames[f'prob_real_{mode}'] = p[0::2]
            frames[f'prob_fake_{mode}'] = p[1::2]
            frames[f'margin_real_{mode}'] = m[0::2]
            frames[f'margin_fake_{mode}'] = m[1::2]
            state['eval_seconds_per_image'] = spi
        vloss = 0.5 * summaries['native']['loss'] + 0.5 * summaries['processed']['loss']
        stop = stopper.update(vloss)
        atomic_csv(out / f'validation_predictions_ve{ve + 1}.csv', frames)
        entry = dict(virtual_epoch=ve + 1, images_seen=state['images_seen'], val_loss=float(vloss),
                     val_native=summaries['native'], val_processed=summaries['processed'],
                     validation_seconds=time.monotonic() - vtick, train_seconds=state.get('ve_train_seconds'),
                     modulation={l: dict(a=float(g.a.detach()), b=float(g.b.detach())) for l, g in model.gsd.items()},
                     early_stopping=dict(stopper.state))
        state['history'] = list(state['history']) + [entry]
        atomic_json(out / 'history.json', state['history'])
        print('[validation] ' + json.dumps({k: entry[k] for k in ('virtual_epoch', 'images_seen', 'val_loss')}) +
              f" native BA={summaries['native']['balanced_accuracy']:.4f} AUC={summaries['native']['auc']:.4f}"
              f" fake_recall={summaries['native']['fake_recall']:.4f} | processed BA="
              f"{summaries['processed']['balanced_accuracy']:.4f}", flush=True)
        payload = evaluator_payload(model, cfg, signature, ve, summaries, vloss, state['images_seen'])
        if cfg['save_virtual_epoch_checkpoints']:
            save_and_verify_payload(out / f'checkpoint_ve{ve + 1}.pt', payload, cfg)
        if state['best_val_loss'] is None or vloss < state['best_val_loss']:
            state.update(best_val_loss=float(vloss), best_ve=ve + 1)
            save_and_verify_payload(out / 'best.pt', payload, cfg)
        del payload
        state.update(ve=ve + 1, phase='train', step=0)
        if stop or state['ve'] >= n_ve:
            state['status'] = 'complete'
            state['stop_reason'] = 'early_stopping' if stop else 'all_virtual_epochs_done'
        state = save_last(out, model, optimizer, scaler, stopper, state)
    if state['status'] == 'complete' and not cfg['keep_last_on_complete'] and (out / 'last.pt').is_file():
        (out / 'last.pt').unlink()
        print('[done] Removed last.pt (optimizer state) to keep the Output small; best.pt is kept.', flush=True)
    return finish_report(out, state)


def append_train_log(out, rows):
    if not rows:
        return
    path = out / 'train_log.csv'
    frame = pd.DataFrame(rows)
    if path.is_file():
        frame = pd.concat([pd.read_csv(path), frame], ignore_index=True)
    atomic_csv(path, frame)


def finish_report(out, state):
    report = dict(status=state['status'], stop_reason=state.get('stop_reason'),
                  virtual_epochs_validated=len(state.get('history', [])),
                  images_seen=state['images_seen'], best_virtual_epoch=state.get('best_ve'),
                  best_val_loss=state.get('best_val_loss'), signature=state['signature'],
                  best_checkpoint=str(out / 'best.pt') if (out / 'best.pt').is_file() else None)
    atomic_json(out / 'selected_checkpoint.json', report)
    if state['status'] == 'complete':
        atomic_json(out / 'completion.json', report)
    print('[report] ' + json.dumps(report, indent=1), flush=True)
    return report

## Cấu hình

In [ ]:
SMOKE = False   # True = thử nhanh 64 ID, 2 VE × 2 bước, output riêng *_smoke (KHÔNG phải kết quả thật)
CONFIG = dict(BFREE_TRAIN_DEFAULTS)
CONFIG.update(
    input_root='/kaggle/input',
    bfree_root=None,          # auto: folder có train_list.csv + valid_list.csv + COCO_real_512
    output_root='/kaggle/working/gsd_bfree_samecat_origbg_v1',
    resume_from='auto',       # auto tìm Output lần trước (cùng cấu hình) đã Add Input
    train_branch='SD2.1_inpainted_samecat_origBG',
    time_budget_hours=11.0,
    smoke=SMOKE,
)
# Chỉ điền khi auto báo lỗi không tìm thấy / nhiều nguồn:
# CONFIG['bfree_root'] = '/kaggle/input/datasets/baopv051/b-free-training-data'
# CONFIG['resume_from'] = '/kaggle/input/<output-lần-trước>/gsd_bfree_samecat_origbg_v1'
# KHÔNG đổi các tham số model/augmentation: chúng phải giống GSD-aug-Tiny để so sánh được.
print(json.dumps(CONFIG, indent=2))

## Train (tự resume; tự dừng sạch trước giới hạn 12 giờ)

In [ ]:
report = run_bfree_training(CONFIG)

## Tóm tắt

In [ ]:
out = Path(CONFIG['output_root'] + ('_smoke' if SMOKE else ''))
print('STATUS:', json.loads((out / 'run_state.json').read_text())['status'])
print(json.dumps(json.loads((out / 'split_summary.json').read_text()), indent=1))
history = json.loads((out / 'history.json').read_text()) if (out / 'history.json').is_file() else []
rows = []
for e in history:
    for view in ('native', 'processed'):
        v = e['val_' + view]
        rows.append(dict(VE=e['virtual_epoch'], images_seen=e['images_seen'], view=view, val_loss=e['val_loss'],
                         loss=v['loss'], BA=v['balanced_accuracy'], AUC=v['auc'], fake_recall=v['fake_recall'],
                         real_recall=v['real_recall'],
                         **{f'fake_recall_{g}': v['by_area'].get(g, {}).get('fake_recall') for g in ('small', 'medium', 'large')}))
if rows:
    display(pd.DataFrame(rows))
if (out / 'train_log.csv').is_file():
    log = pd.read_csv(out / 'train_log.csv')
    display(log.tail(10))
print(json.dumps(json.loads((out / 'selected_checkpoint.json').read_text()), indent=1))
print('Files:', sorted(p.name for p in out.iterdir()))